<a href="https://colab.research.google.com/github/Yisihaq01/breast-histology-calibration/blob/main/Notebooks/05_efficientnet_b0_formal_development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Stage 5 — EfficientNet-B0 Formal Development

## Research Objective

Evaluate whether EfficientNet-B0 produces different discrimination and probability-calibration characteristics from ResNet-18 on the same patient-grouped breast histopathology Development dataset.

## Experimental Design

- Dataset: BreaKHis 40× Development partition
- Development patients: 49
- Development images: 1,133
- Validation: three frozen patient-grouped folds
- Architecture: ImageNet-pretrained EfficientNet-B0
- Input patches: 224 × 224 pixels
- Preprocessing: identical to the ResNet-18 experiment
- Image-level aggregation: sigmoid of mean patch logit
- Maximum training duration: five epochs per fold
- Checkpoint selection: highest validation image-level AUROC, with earliest-epoch tie breaking

## Evaluation

The primary comparison will include:

1. AUROC
2. AUPRC
3. Balanced accuracy
4. Expected Calibration Error (10 equal-width bins)
5. Brier score
6. Negative log-likelihood

The subsequent analysis will include reliability diagrams, calibration diagnostics, and patient-level uncertainty estimates.

## Data Separation

All architecture development and comparisons will use Development patients only.

The 16 Calibration patients and 16 Test patients remain untouched.

External BACH and BRACS datasets will not be used for training or architecture selection.

## Reproducibility

The experiment will reuse the frozen Development manifests, patient folds, preprocessing decisions, and evaluation procedures from the completed ResNet-18 experiment.

Architecture-specific training differences will be documented explicitly before training.


In [1]:

# ============================================================
# Restore repository and verify Development protocol
# ============================================================

from pathlib import Path
import subprocess
import json
import pandas as pd
import numpy as np
import torch
import torchvision

# ------------------------------------------------------------
# 1. Repository setup
# ------------------------------------------------------------

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

REPO_URL = (
    "https://github.com/"
    "Yisihaq01/breast-histology-calibration.git"
)

if not REPO_ROOT.is_dir():

    print("Cloning GitHub repository...")

    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_ROOT)],
        check=True
    )

else:
    print("Repository already exists.")

assert (REPO_ROOT / ".git").is_dir()

# Do not automatically pull or overwrite local changes.
commit = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"],
    text=True
).strip()

print("Git commit:", commit)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print(
    "CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

# ------------------------------------------------------------
# 2. Verify saved Development manifest
# ------------------------------------------------------------

manifest_path = (
    REPO_ROOT /
    "Metadata" /
    "breakhis_development_all_patch_manifest.csv"
)

assert manifest_path.is_file(), (
    f"Development manifest missing: {manifest_path}"
)

manifest = pd.read_csv(
    manifest_path,
    dtype={"patient_id": str}
)

required_columns = {
    "filename",
    "patient_id",
    "label",
    "development_fold"
}

assert required_columns.issubset(
    manifest.columns
)

assert len(manifest) == 17895
assert manifest["filename"].nunique() == 1133
assert manifest["patient_id"].nunique() == 49

# ------------------------------------------------------------
# 3. Verify frozen fold assignments
# ------------------------------------------------------------

assert (
    manifest.groupby("patient_id")[
        "development_fold"
    ].nunique() == 1
).all()

fold_summary = (
    manifest.groupby("development_fold")
    .agg(
        patients=("patient_id", "nunique"),
        images=("filename", "nunique"),
        patches=("filename", "size")
    )
    .reset_index()
    .sort_values("development_fold")
)

print("\n" + "=" * 65)
print("FROZEN DEVELOPMENT FOLD VERIFICATION")
print("=" * 65)

display(fold_summary)

assert (
    fold_summary["patients"].tolist()
    == [17, 16, 16]
)

assert (
    fold_summary["images"].tolist()
    == [372, 431, 330]
)

assert (
    fold_summary["patches"].tolist()
    == [5861, 6810, 5224]
)

# ------------------------------------------------------------
# 4. Verify ResNet-18 comparison results
# ------------------------------------------------------------

RESNET_RESULTS = (
    REPO_ROOT /
    "results" /
    "formal_development" /
    "resnet18" /
    "cross_fold_analysis"
)

comparison_path = (
    RESNET_RESULTS /
    "resnet18_three_fold_comparison.csv"
)

pooled_path = (
    RESNET_RESULTS /
    "resnet18_pooled_oof_metrics.csv"
)

assert comparison_path.is_file()
assert pooled_path.is_file()

resnet_comparison = pd.read_csv(
    comparison_path
)

resnet_pooled = pd.read_csv(
    pooled_path
)

assert len(resnet_comparison) == 3
assert len(resnet_pooled) == 1

print("\nRESNET-18 REFERENCE METRICS")

display(
    resnet_comparison[
        [
            "fold",
            "selected_epoch",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "ece_10",
            "brier",
            "nll"
        ]
    ].round(6)
)

# ------------------------------------------------------------
# 5. Locate reusable training code
# ------------------------------------------------------------

notebook_path = (
    REPO_ROOT /
    "Notebooks" /
    "03_breakhis_formal_development.ipynb"
)

assert notebook_path.is_file()

with open(
    notebook_path,
    "r",
    encoding="utf-8"
) as f:
    original_notebook = json.load(f)

search_terms = [
    "resnet18(",
    "ResNet18_Weights",
    "DataLoader(",
    "AdamW(",
    "BCEWithLogitsLoss",
    "train_one_epoch",
    "def train",
    "class Break",
    "layer4",
    "best_auc",
    "best_auroc"
]

matches = []

for cell_number, cell in enumerate(
    original_notebook["cells"], start=1
):

    if cell["cell_type"] != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    terms_found = [
        term for term in search_terms
        if term.lower() in source.lower()
    ]

    if terms_found:
        matches.append({
            "cell": cell_number,
            "matching_terms": ", ".join(terms_found),
            "first_line": next(
                (
                    line.strip()[:120]
                    for line in source.splitlines()
                    if line.strip()
                ),
                ""
            )
        })

print("\nREUSABLE RESNET-18 TRAINING CODE LOCATIONS")

display(
    pd.DataFrame(matches)
)

# ------------------------------------------------------------
# 6. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STAGE 5 — INITIAL PREFLIGHT")
print("=" * 65)

print("PASS: Repository available.")
print("PASS: All 49 Development patients verified.")
print("PASS: All 1,133 Development images verified.")
print("PASS: Frozen three-fold assignments verified.")
print("PASS: ResNet-18 reference metrics available.")
print("PASS: Previous training notebook available.")

print("\nReady for EfficientNet-B0 model setup.")


Cloning GitHub repository...
Git commit: b3fa310
PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
CUDA available: True
GPU: Tesla T4

FROZEN DEVELOPMENT FOLD VERIFICATION


,development_fold,patients,images,patches
0,1,17,372,5861
1,2,16,431,6810
2,3,16,330,5224



RESNET-18 REFERENCE METRICS


,fold,selected_epoch,auroc,auprc,balanced_accuracy,ece_10,brier,nll
0,1,5,0.992493,0.996657,0.940462,0.074696,0.063208,0.256791
1,2,4,0.956628,0.982234,0.842383,0.099174,0.103134,0.532513
2,3,2,0.998768,0.999624,0.975720,0.037631,0.019652,0.065168



REUSABLE RESNET-18 TRAINING CODE LOCATIONS


,cell,matching_terms,first_line
0,21,class Break,# ============================================...
1,22,DataLoader(,# ============================================...
2,24,layer4,# ============================================...
3,25,"resnet18(, ResNet18_Weights, layer4",# ============================================...
4,26,"AdamW(, BCEWithLogitsLoss, layer4",# ============================================...
5,27,layer4,# ============================================...
6,29,"train_one_epoch, def train",# ============================================...
7,30,"AdamW(, BCEWithLogitsLoss, train_one_epoch, la...",# ============================================...
8,31,best_auc,# ============================================...
9,36,DataLoader(,# ============================================...



STAGE 5 — INITIAL PREFLIGHT
PASS: Repository available.
PASS: All 49 Development patients verified.
PASS: All 1,133 Development images verified.
PASS: Frozen three-fold assignments verified.
PASS: ResNet-18 reference metrics available.
PASS: Previous training notebook available.

Ready for EfficientNet-B0 model setup.



## 5.1 — EfficientNet-B0 Model Configuration

### Objective

Construct an ImageNet-pretrained EfficientNet-B0 classifier for binary breast histopathology classification.

### Architecture

The final ImageNet classification layer is replaced by a single-output linear layer for binary classification.

The earlier feature extraction blocks remain frozen. The final feature blocks and classification head are trainable.

Batch Normalization running statistics will remain frozen during fine-tuning, matching the stabilization approach used in our ResNet-18 experiment.

### Verification

Before training, verify:

- Pretrained model loads successfully.
- Input shape is [batch, 3, 224, 224].
- Output shape is [batch, 1].
- Only the intended parameters require gradients.
- Batch Normalization running statistics are frozen.
- CUDA execution works.

This is an architecture verification step. No histopathology data or model training is involved.

In [2]:

# ============================================================
# EfficientNet-B0 architecture verification
# ============================================================

import torch
import torch.nn as nn

from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)

torch.manual_seed(631)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# 1. Define architecture
# ------------------------------------------------------------

def build_efficientnet_b0():

    weights = EfficientNet_B0_Weights.IMAGENET1K_V1

    # Pretrained weights are required.
    # Do not silently fall back to random initialization.
    model = efficientnet_b0(weights=weights)

    # Freeze all pretrained parameters first.
    for parameter in model.parameters():
        parameter.requires_grad = False

    # Replace ImageNet classifier with binary output.
    in_features = model.classifier[-1].in_features

    model.classifier[-1] = nn.Linear(
        in_features,
        1
    )

    # Fine-tune the final feature blocks.
    for parameter in model.features[7].parameters():
        parameter.requires_grad = True

    for parameter in model.features[8].parameters():
        parameter.requires_grad = True

    return model


# ------------------------------------------------------------
# 2. Freeze Batch Normalization running statistics
# ------------------------------------------------------------

def freeze_efficientnet_bn_statistics(model):

    for module in model.modules():

        if isinstance(
            module,
            (
                nn.BatchNorm1d,
                nn.BatchNorm2d,
                nn.BatchNorm3d,
                nn.SyncBatchNorm
            )
        ):
            module.eval()


def set_efficientnet_training_mode(model):

    model.train()

    # Calling model.train() reactivates BatchNorm.
    # Explicitly freeze its running statistics again.
    freeze_efficientnet_bn_statistics(model)


# ------------------------------------------------------------
# 3. Initialize model
# ------------------------------------------------------------

efficientnet_model = build_efficientnet_b0()

efficientnet_model = efficientnet_model.to(DEVICE)

total_parameters = sum(
    parameter.numel()
    for parameter in efficientnet_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in efficientnet_model.parameters()
    if parameter.requires_grad
)

frozen_parameters = (
    total_parameters - trainable_parameters
)

print("=" * 65)
print("EFFICIENTNET-B0 — MODEL ARCHITECTURE")
print("=" * 65)

print("Device:", DEVICE)

print("Total parameters:", f"{total_parameters:,}")

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)

print(
    "Frozen parameters:",
    f"{frozen_parameters:,}"
)

print(
    "Trainable percentage:",
    f"{100 * trainable_parameters / total_parameters:.2f}%"
)


# ------------------------------------------------------------
# 4. Verify trainable parameter names
# ------------------------------------------------------------

trainable_names = [
    name
    for name, parameter in efficientnet_model.named_parameters()
    if parameter.requires_grad
]

assert len(trainable_names) > 0

assert all(
    name.startswith((
        "features.7.",
        "features.8.",
        "classifier."
    ))
    for name in trainable_names
)

print("\nTrainable parameter groups:")

for prefix in [
    "features.7",
    "features.8",
    "classifier"
]:

    count = sum(
        parameter.numel()
        for name, parameter
        in efficientnet_model.named_parameters()

        if parameter.requires_grad
        and name.startswith(prefix)
    )

    print(f"{prefix}: {count:,}")


# ------------------------------------------------------------
# 5. Verify BatchNorm statistics remain frozen
# ------------------------------------------------------------

set_efficientnet_training_mode(
    efficientnet_model
)

bn_modules = [
    module
    for module in efficientnet_model.modules()
    if isinstance(module, nn.BatchNorm2d)
]

assert len(bn_modules) > 0

assert all(
    not module.training
    for module in bn_modules
)

print("\nBatchNorm layers:", len(bn_modules))

print("BatchNorm training mode frozen: True")


# ------------------------------------------------------------
# 6. Verify forward pass on GPU
# ------------------------------------------------------------

efficientnet_model.eval()

dummy_input = torch.zeros(
    1, 3, 224, 224,
    device=DEVICE
)

with torch.no_grad():

    dummy_output = efficientnet_model(
        dummy_input
    )

assert dummy_output.shape == (1, 1)

assert torch.isfinite(
    dummy_output
).all()

print("\nInput shape:", tuple(dummy_input.shape))

print("Output shape:", tuple(dummy_output.shape))


# ------------------------------------------------------------
# 7. Release test tensors
# ------------------------------------------------------------

del dummy_input, dummy_output

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# 8. Final checks
# ------------------------------------------------------------

assert total_parameters > trainable_parameters
assert trainable_parameters > 0

print("\n" + "=" * 65)
print("EFFICIENTNET-B0 ARCHITECTURE PREFLIGHT")
print("=" * 65)

print("PASS: ImageNet-pretrained model initialized.")
print("PASS: Binary classification head configured.")
print("PASS: Intended layers selected for fine-tuning.")
print("PASS: BatchNorm running statistics frozen.")
print("PASS: 224x224 forward pass verified.")
print("PASS: EfficientNet-B0 ready for training setup.")


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 113MB/s] 


EFFICIENTNET-B0 — MODEL ARCHITECTURE
Device: cuda
Total parameters: 4,008,829
Trainable parameters: 1,130,673
Frozen parameters: 2,878,156
Trainable percentage: 28.20%

Trainable parameter groups:
features.7: 717,232
features.8: 412,160
classifier: 1,281

BatchNorm layers: 49
BatchNorm training mode frozen: True

Input shape: (1, 3, 224, 224)
Output shape: (1, 1)

EFFICIENTNET-B0 ARCHITECTURE PREFLIGHT
PASS: ImageNet-pretrained model initialized.
PASS: Binary classification head configured.
PASS: Intended layers selected for fine-tuning.
PASS: BatchNorm running statistics frozen.
PASS: 224x224 forward pass verified.
PASS: EfficientNet-B0 ready for training setup.



## 5.2 — Training Configuration and Reproducibility Audit

### Objective

Inspect the completed ResNet-18 training implementation before adapting it to EfficientNet-B0.

### Configuration Requirements

We will preserve:

- The same frozen patient-level Development folds.
- The same patch extraction and image normalization.
- The same training-only augmentation.
- The same class-weighting approach.
- The same maximum of five training epochs.
- The same image-level AUROC checkpoint-selection rule.
- The same mean-logit image aggregation.
- The same evaluation metric definitions.

### Architecture-Specific Differences

EfficientNet-B0 has a different layer structure and parameter count from ResNet-18.

Its final feature blocks and binary classification head will be fine-tuned, with Batch Normalization running statistics frozen.

We will document these differences and inspect the original optimizer settings before running any training experiments.

### Data Separation

Only Development training and validation folds may be accessed during model development.

Calibration, Test, BACH, and BRACS outcomes remain excluded.


In [3]:

# ============================================================
# Audit original ResNet-18 training implementation
# ============================================================

import json
from pathlib import Path

notebook_path = (
    REPO_ROOT /
    "Notebooks" /
    "03_breakhis_formal_development.ipynb"
)

assert notebook_path.is_file()

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# Cell numbers are 1-based, matching the Stage 5
# preflight report from Cell 2.
target_cells = [21, 22, 25, 26, 27, 29, 30, 31]

search_terms = [
    "DataLoader",
    "Dataset",
    "batch_size",
    "num_workers",
    "optimizer",
    "AdamW",
    "weight_decay",
    "pos_weight",
    "BCEWithLogitsLoss",
    "learning_rate",
    "lr=",
    "train_one_epoch",
    "train_model",
    "model.train",
    "autocast",
    "GradScaler",
    "scheduler",
    "augmentation",
    "RandomHorizontalFlip",
    "RandomVerticalFlip",
    "Normalize",
    "epoch",
    "AUROC",
    "best_auroc",
    "state_dict"
]

for cell_number in target_cells:

    cell = notebook["cells"][cell_number - 1]

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))
    lines = source.splitlines()

    # Find configuration-relevant source lines.
    matched_indices = [
        i for i, line in enumerate(lines)
        if any(
            term.lower() in line.lower()
            for term in search_terms
        )
    ]

    # Include a few neighboring lines for context.
    indices = set()

    for i in matched_indices:
        for j in range(
            max(0, i - 2),
            min(len(lines), i + 4)
        ):
            indices.add(j)

    # Display no more than 65 lines per code cell.
    selected = sorted(indices)[:65]

    print("\n" + "=" * 68)
    print(
        f"ORIGINAL NOTEBOOK — CODE CELL {cell_number}"
    )
    print("=" * 68)

    if not selected:
        print("No matching configuration lines.")
        continue

    previous = None

    for index in selected:

        if (
            previous is not None
            and index > previous + 1
        ):
            print("    ...")

        print(
            f"{index + 1:>4}: {lines[index]}"
        )

        previous = index

    if len(indices) > len(selected):
        print(
            f"... {len(indices) - len(selected)} "
            "additional relevant lines omitted"
        )

print("\n" + "=" * 68)
print("TRAINING CONFIGURATION AUDIT COMPLETE")
print("=" * 68)

print("PASS: Original Notebook 03 loaded.")
print("PASS: Training-related code sections inspected.")
print("PASS: No models trained or checkpoints modified.")



ORIGINAL NOTEBOOK — CODE CELL 21
   1: 
   2: # ============================================================
   3: # BreakHis patch Dataset
   4: # ============================================================
   5: 
   6: from pathlib import Path
   7: 
   8: from torch.utils.data import Dataset, DataLoader
   9: 
  10: 
  11: class BreakHisPatchDataset(Dataset):
  12: 
  13:     def __init__(
  14:         self,
  15:         manifest,
  16:         dataset_root,
  17:         transform=None
  18:     ):
  19: 
    ...
  24:         )
  25: 
  26:         self.dataset_root = Path(
  27:             dataset_root
  28:         )
  29: 
  30:         self.transform = transform
    ...
  55:         ).issubset({0, 1}):
  56:             raise ValueError(
  57:                 "Dataset labels must be binary."
  58:             )
  59: 
  60:     def __len__(self):
    ...
  67: 
  68:         image_path = (
  69:             self.dataset_root /
  70:             row["relative_path"]
  71:


## 5.3 — Training Configuration Transfer

### Objective

Transfer the original ResNet-18 training protocol to EfficientNet-B0 while preserving all applicable preprocessing and evaluation decisions.

### Confirmed Shared Settings

- Batch size: 32
- DataLoader workers: 2
- Optimizer: AdamW
- Loss: BCEWithLogitsLoss with training-fold class weighting
- Mixed precision: FP16 autocast and gradient scaling
- Validation: patient-grouped, image-level evaluation
- Checkpoint selection: maximum validation image AUROC

### Architecture-Specific Adaptations

The EfficientNet-B0 optimizer will update the final feature blocks and classification head rather than ResNet-18's layer4 and fc.

We will verify the original learning rates, weight decay, image transformations, and data path before training.

No training will occur during this audit.


In [4]:

# ============================================================
# Extract exact ResNet-18 training configuration
# ============================================================

import ast
import json
from pathlib import Path

source_notebook_path = (
    REPO_ROOT /
    "Notebooks" /
    "03_breakhis_formal_development.ipynb"
)

with open(source_notebook_path, "r", encoding="utf-8") as f:
    source_notebook = json.load(f)

# Definitions we need to preserve or adapt
target_names = {
    "TRAIN_CONFIG",
    "DATASET_ROOT",
    "train_transform",
    "val_transform",
    "loader_generator",
    "seed_worker"
}

found_names = set()

print("=" * 65)
print("ORIGINAL TRAINING CONFIGURATION DEFINITIONS")
print("=" * 65)

for cell_number, cell in enumerate(
    source_notebook["cells"], start=1
):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:

        names = set()

        if isinstance(node, (ast.Assign, ast.AnnAssign)):

            targets = (
                node.targets
                if isinstance(node, ast.Assign)
                else [node.target]
            )

            for target in targets:
                if isinstance(target, ast.Name):
                    names.add(target.id)

        elif isinstance(
            node, (ast.FunctionDef, ast.AsyncFunctionDef)
        ):
            names.add(node.name)

        matches = names & target_names

        if not matches:
            continue

        # Prefer the first definition of each object.
        new_matches = matches - found_names

        if not new_matches:
            continue

        found_names.update(new_matches)

        definition = ast.get_source_segment(
            source, node
        )

        print("\n" + "-" * 65)
        print(
            f"Notebook cell {cell_number}: "
            f"{', '.join(sorted(new_matches))}"
        )
        print("-" * 65)

        if definition:
            lines = definition.splitlines()

            for line in lines[:65]:
                print(line)

            if len(lines) > 65:
                print(
                    f"... {len(lines) - 65} "
                    "additional lines omitted"
                )

print("\n" + "=" * 65)
print("DEFINITION INVENTORY")
print("=" * 65)

print("Found:", sorted(found_names))
print("Not found:", sorted(target_names - found_names))

# ------------------------------------------------------------
# Inspect reusable dataset source and environment
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("DATASET AVAILABILITY")
print("=" * 65)

candidate_roots = [
    Path("/root/.cache/kagglehub/datasets/ambarish/breakhis"),
    Path("/content/breakhis"),
    Path("/content/datasets"),
    REPO_ROOT / "data"
]

for path in candidate_roots:
    print(
        f"{path}:",
        "EXISTS" if path.exists() else "not present"
    )

print("\nPASS: Configuration definitions inspected.")
print("PASS: Dataset locations checked.")
print("PASS: No training executed.")


ORIGINAL TRAINING CONFIGURATION DEFINITIONS

-----------------------------------------------------------------
Notebook cell 20: train_transform
-----------------------------------------------------------------
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    RandomRightAngleRotation(),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

-----------------------------------------------------------------
Notebook cell 20: val_transform
-----------------------------------------------------------------
val_transform = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

-----------------------------------------------------------------
Notebook cell 22: seed_worker
-----------------------------------------------------------------



## 5.4 — Restore and Verify Source Images

### Objective

Restore access to the original BreaKHis images while preserving the frozen Development manifest.

### Data Source

The BreaKHis dataset is retrieved using KaggleHub from the previously selected dataset source.

### Verification

Before training:

1. Resolve the local dataset directory.
2. Confirm that all 1,133 Development source images exist.
3. Preserve the original 49 patients and three validation folds.
4. Verify that a source image can be opened successfully.
5. Keep original image files outside the GitHub repository.

The Development manifest remains the authoritative record of which images and patches may be used.

Calibration, Test, BACH, and BRACS images must not enter the Development training or validation DataLoaders.

No model training is performed during this step.


In [5]:

# ============================================================
# Restore BreaKHis images and verify Development coverage
# ============================================================

from pathlib import Path
from PIL import Image
import pandas as pd
import importlib.util
import subprocess
import sys

# ------------------------------------------------------------
# 1. Verify frozen manifest
# ------------------------------------------------------------

assert len(manifest) == 17895

required = {
    "filename",
    "relative_path",
    "patient_id",
    "label",
    "development_fold"
}

assert required.issubset(manifest.columns)

development_images = (
    manifest[
        ["filename", "relative_path", "patient_id"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

assert len(development_images) == 1133
assert development_images["filename"].is_unique
assert development_images["patient_id"].nunique() == 49

# Normalize relative paths for Colab/Linux
development_images["relative_path"] = (
    development_images["relative_path"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
)

relative_paths = [
    Path(value)
    for value in development_images["relative_path"]
]

assert all(
    not path.is_absolute()
    and ".." not in path.parts
    for path in relative_paths
)

sample_relative = relative_paths[0]

print("Manifest sample:", sample_relative)

# ------------------------------------------------------------
# 2. Install KaggleHub if necessary
# ------------------------------------------------------------

if importlib.util.find_spec("kagglehub") is None:
    subprocess.run(
        [
            sys.executable,
            "-m", "pip", "install",
            "-q", "kagglehub"
        ],
        check=True
    )

import kagglehub

# ------------------------------------------------------------
# 3. Retrieve the original dataset source
# ------------------------------------------------------------

print("\nRestoring BreaKHis source dataset...")

download_root = Path(
    kagglehub.dataset_download("ambarish/breakhis")
).resolve()

assert download_root.is_dir()

print("Downloaded dataset location:", download_root)

# ------------------------------------------------------------
# 4. Locate the directory used by the old Dataset class
# ------------------------------------------------------------

def resolve_dataset_root(download_root, relative_path):

    # First try the returned directory directly.
    candidates = [download_root]

    # Some archives contain an extra top-level directory.
    candidates.extend(
        path for path in download_root.iterdir()
        if path.is_dir()
    )

    for candidate in candidates:
        if (candidate / relative_path).is_file():
            return candidate

    # Resolve deeper nested layouts using a manifest file.
    suffix = tuple(relative_path.parts)

    for match in download_root.rglob(relative_path.name):

        if not match.is_file():
            continue

        if tuple(match.parts[-len(suffix):]) == suffix:

            candidate = match.parents[len(suffix) - 1]

            if (candidate / relative_path).is_file():
                return candidate

    raise FileNotFoundError(
        "Cannot match the Development manifest to "
        f"the downloaded dataset. Example: {relative_path}"
    )


DATASET_ROOT = resolve_dataset_root(
    download_root,
    sample_relative
)

print("\nResolved DATASET_ROOT:")
print(DATASET_ROOT)

# ------------------------------------------------------------
# 5. Verify every Development image exists
# ------------------------------------------------------------

missing_images = []

for relative_path in relative_paths:

    image_path = DATASET_ROOT / relative_path

    if not image_path.is_file():
        missing_images.append(str(relative_path))

assert not missing_images, (
    f"{len(missing_images)} Development images missing. "
    f"Examples: {missing_images[:5]}"
)

# ------------------------------------------------------------
# 6. Verify one image can be decoded
# ------------------------------------------------------------

sample_image_path = DATASET_ROOT / sample_relative

with Image.open(sample_image_path) as image:

    sample_size = image.size
    sample_mode = image.mode

    image.verify()

print("\nSample source image:")
print("Size:", sample_size)
print("Mode:", sample_mode)

# ------------------------------------------------------------
# 7. Final checks
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BREAKHIS DEVELOPMENT DATA RESTORATION")
print("=" * 65)

print("Development patients:", 49)
print("Development images:", len(development_images))
print("Frozen patch records:", len(manifest))
print("Missing Development images:", len(missing_images))

print("\nPASS: BreaKHis dataset accessible.")
print("PASS: DATASET_ROOT resolved.")
print("PASS: All 1,133 Development images located.")
print("PASS: Sample source image verified.")
print("PASS: Frozen Development manifest unchanged.")
print("\nReady for EfficientNet-B0 DataLoader setup.")


Manifest sample: BreaKHis_v1/BreaKHis_v1/histology_slides/breast/benign/SOB/tubular_adenoma/SOB_B_TA_14-13200/40X/SOB_B_TA-14-13200-40-001.png

Restoring BreaKHis source dataset...


100%|██████████| 3.99G/3.99G [00:54<00:00, 78.3MB/s]

Extracting files...


Downloaded dataset location: /root/.cache/kagglehub/datasets/ambarish/breakhis/versions/4

Resolved DATASET_ROOT:
/root/.cache/kagglehub/datasets/ambarish/breakhis/versions/4

Sample source image:
Size: (700, 460)
Mode: RGB

BREAKHIS DEVELOPMENT DATA RESTORATION
Development patients: 49
Development images: 1133
Frozen patch records: 17895
Missing Development images: 0

PASS: BreaKHis dataset accessible.
PASS: DATASET_ROOT resolved.
PASS: All 1,133 Development images located.
PASS: Sample source image verified.
PASS: Frozen Development manifest unchanged.

Ready for EfficientNet-B0 DataLoader setup.



## 5.5 — EfficientNet-B0 Fold 1 Data Preparation

### Objective

Prepare the first EfficientNet-B0 training and validation datasets using the same frozen Development partitions and patch preprocessing as ResNet-18.

### Patient-Level Separation

- Training: Development Folds 2 and 3
- Validation: Development Fold 1
- Expected training patients: 32
- Expected validation patients: 17
- Expected training patches: 12,034
- Expected validation patches: 5,861

### Preprocessing

Reuse the original BreakHisPatchDataset and RandomRightAngleRotation implementations from Notebook 03.

Training augmentation includes random horizontal flipping, random vertical flipping, and random 90-degree rotations.

Validation uses deterministic preprocessing without augmentation.

Both datasets use the same ImageNet normalization.

### Verification

Before model training:

1. Confirm patient separation and exact sample counts.
2. Restore the original dataset and rotation classes.
3. Construct training and validation datasets.
4. Verify that extracted patches have the expected 3 × 224 × 224 tensor dimensions.
5. Verify that validation preprocessing is deterministic.

No model training or optimizer updates are performed in this step.


In [8]:

# ============================================================
# Complete EfficientNet-B0 Fold 1 Data Preparation
# ============================================================

import ast
import json
import random
import math
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from PIL import Image, ImageOps
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.transforms import functional as TF
from IPython.display import display

# ------------------------------------------------------------
# 1. Verify frozen Fold 1 patient separation
# ------------------------------------------------------------

FOLD = 1

train_df = manifest.loc[
    manifest["development_fold"] != FOLD
].copy().reset_index(drop=True)

val_df = manifest.loc[
    manifest["development_fold"] == FOLD
].copy().reset_index(drop=True)

train_patients = set(train_df["patient_id"])
val_patients = set(val_df["patient_id"])

assert train_patients.isdisjoint(val_patients)

assert len(train_patients) == 32
assert len(val_patients) == 17

assert len(train_df) == 12034
assert len(val_df) == 5861

assert train_df["filename"].nunique() == 761
assert val_df["filename"].nunique() == 372

fold_summary = pd.DataFrame([
    {
        "partition": "Training",
        "patients": len(train_patients),
        "images": train_df["filename"].nunique(),
        "patches": len(train_df)
    },
    {
        "partition": "Validation",
        "patients": len(val_patients),
        "images": val_df["filename"].nunique(),
        "patches": len(val_df)
    }
])

print("=" * 65)
print("EFFICIENTNET-B0 — FOLD 1 PARTITIONS")
print("=" * 65)

display(fold_summary)

print("PASS: Patient-grouped Fold 1 split verified.")


# ------------------------------------------------------------
# 2. Read original ResNet-18 training notebook
# ------------------------------------------------------------

reference_path = (
    REPO_ROOT /
    "Notebooks" /
    "03_breakhis_formal_development.ipynb"
)

assert reference_path.is_file()

with open(reference_path, "r", encoding="utf-8") as f:
    reference_nb = json.load(f)


# ------------------------------------------------------------
# 3. Locate original preprocessing definitions
# ------------------------------------------------------------

original_definitions = {}
original_constants = {}

for cell in reference_nb["cells"]:

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:

        if isinstance(
            node,
            (ast.FunctionDef, ast.ClassDef)
        ):
            original_definitions[node.name] = node

        elif isinstance(node, ast.Assign):

            if len(node.targets) != 1:
                continue

            target = node.targets[0]

            if not isinstance(target, ast.Name):
                continue

            try:
                original_constants[target.id] = (
                    ast.literal_eval(node.value)
                )
            except (ValueError, TypeError, SyntaxError,
                    MemoryError, RecursionError):
                pass


# ------------------------------------------------------------
# 4. Restore original classes and helper functions
# ------------------------------------------------------------

required_names = {
    "RandomRightAngleRotation",
    "BreakHisPatchDataset",
    "resample_to_target_mpp"
}

missing_definitions = (
    required_names - original_definitions.keys()
)

assert not missing_definitions, (
    "Missing original definitions: "
    f"{sorted(missing_definitions)}"
)

# Include any other original helper functions called
# by these definitions.
dependencies = set(required_names)

while True:

    expanded = set(dependencies)

    for name in dependencies:

        node = original_definitions[name]

        referenced_names = {
            child.id
            for child in ast.walk(node)
            if isinstance(child, ast.Name)
            and isinstance(child.ctx, ast.Load)
        }

        expanded.update(
            referenced_names
            & original_definitions.keys()
        )

    if expanded == dependencies:
        break

    dependencies = expanded


# Restore any directly referenced literal constants
# from the original notebook.
for name in dependencies:

    node = original_definitions[name]

    for child in ast.walk(node):

        if not isinstance(child, ast.Name):
            continue

        if child.id in original_constants:
            if child.id not in globals():
                globals()[child.id] = (
                    original_constants[child.id]
                )


# Compile original definitions without running
# the old notebook's training code.
definition_nodes = [
    original_definitions[name]
    for name in sorted(dependencies)
]

definition_module = ast.fix_missing_locations(
    ast.Module(
        body=definition_nodes,
        type_ignores=[]
    )
)

exec(
    compile(
        definition_module,
        filename="original_breakhis_preprocessing",
        mode="exec"
    ),
    globals()
)

print("\nRestored original preprocessing definitions:")

for name in sorted(dependencies):
    print(" -", name)

assert callable(resample_to_target_mpp)
assert "BreakHisPatchDataset" in globals()
assert "RandomRightAngleRotation" in globals()

print("\nPASS: Original preprocessing restored.")


# ------------------------------------------------------------
# 5. Recreate original transformations
# ------------------------------------------------------------

efficientnet_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    RandomRightAngleRotation(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

efficientnet_val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("PASS: Original augmentation and normalization restored.")


# ------------------------------------------------------------
# 6. Construct Fold 1 patch datasets
# ------------------------------------------------------------

efficientnet_train_dataset = BreakHisPatchDataset(
    manifest=train_df,
    dataset_root=DATASET_ROOT,
    transform=efficientnet_train_transform
)

efficientnet_val_dataset = BreakHisPatchDataset(
    manifest=val_df,
    dataset_root=DATASET_ROOT,
    transform=efficientnet_val_transform
)

assert len(efficientnet_train_dataset) == 12034
assert len(efficientnet_val_dataset) == 5861

print("PASS: Fold 1 datasets constructed.")


# ------------------------------------------------------------
# 7. Verify patch extraction and tensor dimensions
# ------------------------------------------------------------

def get_image_tensor(sample):

    if isinstance(sample, torch.Tensor):
        candidates = [sample]

    elif isinstance(sample, (tuple, list)):
        candidates = [
            value for value in sample
            if isinstance(value, torch.Tensor)
            and value.ndim == 3
        ]

    elif isinstance(sample, dict):
        candidates = [
            value for value in sample.values()
            if isinstance(value, torch.Tensor)
            and value.ndim == 3
        ]

    else:
        raise TypeError(
            f"Unexpected sample type: {type(sample)}"
        )

    assert len(candidates) == 1

    return candidates[0]


train_sample = efficientnet_train_dataset[0]
val_sample = efficientnet_val_dataset[0]

train_image = get_image_tensor(train_sample)
val_image = get_image_tensor(val_sample)

assert train_image.shape == (3, 224, 224)
assert val_image.shape == (3, 224, 224)

assert torch.isfinite(train_image).all()
assert torch.isfinite(val_image).all()

# Validation must produce the same image every time.
val_image_repeat = get_image_tensor(
    efficientnet_val_dataset[0]
)

assert torch.equal(
    val_image,
    val_image_repeat
)

print("\nTraining patch:", tuple(train_image.shape))
print("Validation patch:", tuple(val_image.shape))

print("PASS: Original patch extraction verified.")
print("PASS: Validation preprocessing deterministic.")


# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STAGE 5 — FOLD 1 DATASET PREFLIGHT")
print("=" * 65)

print("Training patients:", len(train_patients))
print("Validation patients:", len(val_patients))

print("Training patches:", len(efficientnet_train_dataset))
print("Validation patches:", len(efficientnet_val_dataset))

print("\nPASS: Frozen patient separation verified.")
print("PASS: Original preprocessing dependencies restored.")
print("PASS: Original augmentation preserved.")
print("PASS: Original patch Dataset reused.")
print("PASS: Input tensors verified.")
print("PASS: No model training performed.")


EFFICIENTNET-B0 — FOLD 1 PARTITIONS


,partition,patients,images,patches
0,Training,32,761,12034
1,Validation,17,372,5861


PASS: Patient-grouped Fold 1 split verified.

Restored original preprocessing definitions:
 - BreakHisPatchDataset
 - RandomRightAngleRotation
 - resample_to_target_mpp

PASS: Original preprocessing restored.
PASS: Original augmentation and normalization restored.
PASS: Fold 1 datasets constructed.

Training patch: (3, 224, 224)
Validation patch: (3, 224, 224)
PASS: Original patch extraction verified.
PASS: Validation preprocessing deterministic.

STAGE 5 — FOLD 1 DATASET PREFLIGHT
Training patients: 32
Validation patients: 17
Training patches: 12034
Validation patches: 5861

PASS: Frozen patient separation verified.
PASS: Original preprocessing dependencies restored.
PASS: Original augmentation preserved.
PASS: Original patch Dataset reused.
PASS: Input tensors verified.
PASS: No model training performed.



## 5.6 — EfficientNet-B0 Fold 1 Training Configuration

### Objective

Prepare reproducible training and validation DataLoaders and configure the EfficientNet-B0 optimization procedure.

### DataLoaders

- Training patches: 12,034
- Validation patches: 5,861
- Batch size: 32
- Workers: 2
- Training shuffle: enabled
- Validation shuffle: disabled
- Random seed: 631

The training and validation datasets use the original preprocessing from Notebook 03.

### Loss Function

Binary cross-entropy with logits is used for classification.

The malignant class is labeled 1.

Positive-class weighting is calculated exclusively from the training fold:

positive weight = benign training patches / malignant training patches

### Optimization

The AdamW optimizer uses:

- Final feature blocks: learning rate 0.0001
- Classification head: learning rate 0.0005
- Weight decay: 0.0001

The training duration will be five epochs, and the best checkpoint will be selected using validation image-level AUROC.

### Safeguards

No model training, weight updates, or checkpoint selection occurs in this setup step.


In [9]:

# ============================================================
# EfficientNet-B0 Fold 1 DataLoaders
# ============================================================

import random
import numpy as np
import torch

from torch.utils.data import DataLoader

SEED = 631
BATCH_SIZE = 32
NUM_WORKERS = 2

# ------------------------------------------------------------
# 1. Reproducibility settings
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def seed_worker(worker_id):

    worker_seed = (
        torch.initial_seed() % (2 ** 32)
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

# ------------------------------------------------------------
# 2. Training DataLoader
# ------------------------------------------------------------

efficientnet_train_loader = DataLoader(
    efficientnet_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    generator=loader_generator
)

# ------------------------------------------------------------
# 3. Validation DataLoader
# ------------------------------------------------------------

efficientnet_val_loader = DataLoader(
    efficientnet_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

# ------------------------------------------------------------
# 4. Verify batch structure
# ------------------------------------------------------------

train_batch = next(
    iter(efficientnet_train_loader)
)

val_batch = next(
    iter(efficientnet_val_loader)
)

def inspect_batch(batch, name):

    print(f"\n{name} batch structure:")

    if isinstance(batch, dict):
        for key, value in batch.items():
            if isinstance(value, torch.Tensor):
                print(
                    key,
                    tuple(value.shape),
                    value.dtype
                )
            else:
                print(key, type(value).__name__)

        elements = list(batch.values())

    elif isinstance(batch, (tuple, list)):
        elements = batch

        for i, value in enumerate(elements):
            if isinstance(value, torch.Tensor):
                print(
                    f"Element {i}:",
                    tuple(value.shape),
                    value.dtype
                )
            else:
                print(
                    f"Element {i}:",
                    type(value).__name__
                )

    else:
        raise TypeError(
            f"Unexpected batch type: {type(batch)}"
        )

    image_tensors = [
        value for value in elements
        if isinstance(value, torch.Tensor)
        and value.ndim == 4
    ]

    assert len(image_tensors) == 1

    images = image_tensors[0]

    assert images.shape == (
        BATCH_SIZE, 3, 224, 224
    )

    assert torch.isfinite(images).all()

    return images


train_images = inspect_batch(
    train_batch, "Training"
)

val_images = inspect_batch(
    val_batch, "Validation"
)

# ------------------------------------------------------------
# 5. Verify loader coverage
# ------------------------------------------------------------

assert len(efficientnet_train_dataset) == 12034
assert len(efficientnet_val_dataset) == 5861

expected_train_batches = (
    12034 + BATCH_SIZE - 1
) // BATCH_SIZE

expected_val_batches = (
    5861 + BATCH_SIZE - 1
) // BATCH_SIZE

assert (
    len(efficientnet_train_loader)
    == expected_train_batches
)

assert (
    len(efficientnet_val_loader)
    == expected_val_batches
)

print("\n" + "=" * 65)
print("EFFICIENTNET-B0 — FOLD 1 DATALOADERS")
print("=" * 65)

print("Training batches:", len(efficientnet_train_loader))
print("Validation batches:", len(efficientnet_val_loader))

print("\nPASS: Training DataLoader initialized.")
print("PASS: Validation DataLoader initialized.")
print("PASS: Batch dimensions and finite values verified.")
print("PASS: Batch counts verified.")
print("PASS: No model training performed.")

# ------------------------------------------------------------
# 6. Clean up inspection batch objects
# ------------------------------------------------------------

del train_batch, val_batch, train_images, val_images



Training batch structure:
image (32, 3, 224, 224) torch.float32
label (32,) torch.float32
patient_id list
filename list
patch_index (32,) torch.int64

Validation batch structure:
image (32, 3, 224, 224) torch.float32
label (32,) torch.float32
patient_id list
filename list
patch_index (32,) torch.int64

EFFICIENTNET-B0 — FOLD 1 DATALOADERS
Training batches: 377
Validation batches: 184

PASS: Training DataLoader initialized.
PASS: Validation DataLoader initialized.
PASS: Batch dimensions and finite values verified.
PASS: Batch counts verified.
PASS: No model training performed.


In [10]:

# ============================================================
# EfficientNet-B0 Fold 1 Loss and Optimizer
# ============================================================

import torch
import torch.nn as nn

# ------------------------------------------------------------
# 1. Record architecture-specific training configuration
# ------------------------------------------------------------

EFFICIENTNET_CONFIG = {
    "architecture": "efficientnet_b0",
    "pretrained_weights": "IMAGENET1K_V1",
    "finetune_layers": [
        "features.7",
        "features.8",
        "classifier"
    ],
    "freeze_batchnorm_statistics": True,
    "batch_size": 32,
    "epochs": 5,
    "lr_features": 1e-4,
    "lr_classifier": 5e-4,
    "weight_decay": 1e-4,
    "seed": 631,
    "selection_metric": "image_auroc",
    "validation_folds": [1, 2, 3]
}

# ------------------------------------------------------------
# 2. Calculate training-fold class weights
# ------------------------------------------------------------

train_label_counts = (
    train_df["label"]
    .value_counts()
    .to_dict()
)

n_benign = int(
    train_label_counts.get(0, 0)
)

n_malignant = int(
    train_label_counts.get(1, 0)
)

assert n_benign > 0
assert n_malignant > 0

pos_weight_value = (
    n_benign / n_malignant
)

device = DEVICE

pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

# ------------------------------------------------------------
# 3. Configure EfficientNet-B0 optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    [
        {
            "params": list(
                efficientnet_model.features[7].parameters()
            ) + list(
                efficientnet_model.features[8].parameters()
            ),
            "lr": EFFICIENTNET_CONFIG["lr_features"]
        },
        {
            "params": efficientnet_model.classifier.parameters(),
            "lr": EFFICIENTNET_CONFIG["lr_classifier"]
        }
    ],
    weight_decay=EFFICIENTNET_CONFIG["weight_decay"]
)

# ------------------------------------------------------------
# 4. Verify trainable parameters
# ------------------------------------------------------------

optimizer_param_ids = [
    id(parameter)
    for group in optimizer.param_groups
    for parameter in group["params"]
]

trainable_param_ids = [
    id(parameter)
    for parameter in efficientnet_model.parameters()
    if parameter.requires_grad
]

assert len(optimizer_param_ids) == len(
    set(optimizer_param_ids)
), "Duplicate parameters in optimizer."

assert set(optimizer_param_ids) == set(
    trainable_param_ids
), "Optimizer parameters do not match trainable layers."

assert len(optimizer.param_groups) == 2

assert optimizer.param_groups[0]["lr"] == 1e-4
assert optimizer.param_groups[1]["lr"] == 5e-4

assert optimizer.defaults["weight_decay"] == 1e-4

# ------------------------------------------------------------
# 5. Display training configuration
# ------------------------------------------------------------

print("=" * 65)
print("EFFICIENTNET-B0 — FOLD 1 OPTIMIZATION")
print("=" * 65)

print("Benign training patches:", n_benign)
print("Malignant training patches:", n_malignant)

print(
    "Positive class weight:",
    round(pos_weight_value, 6)
)

print("\nOptimizer:", type(optimizer).__name__)

print(
    "Feature blocks LR:",
    optimizer.param_groups[0]["lr"]
)

print(
    "Classifier LR:",
    optimizer.param_groups[1]["lr"]
)

print(
    "Weight decay:",
    optimizer.defaults["weight_decay"]
)

print(
    "Epochs:",
    EFFICIENTNET_CONFIG["epochs"]
)

print("\nPASS: Class weighting computed from training fold only.")
print("PASS: BCEWithLogitsLoss configured.")
print("PASS: AdamW optimizer configured.")
print("PASS: All intended trainable parameters included.")
print("PASS: No optimizer steps or weight updates performed.")


EFFICIENTNET-B0 — FOLD 1 OPTIMIZATION
Benign training patches: 3301
Malignant training patches: 8733
Positive class weight: 0.377992

Optimizer: AdamW
Feature blocks LR: 0.0001
Classifier LR: 0.0005
Weight decay: 0.0001
Epochs: 5

PASS: Class weighting computed from training fold only.
PASS: BCEWithLogitsLoss configured.
PASS: AdamW optimizer configured.
PASS: All intended trainable parameters included.
PASS: No optimizer steps or weight updates performed.



## 5.7 — EfficientNet-B0 Gradient Smoke Test

### Objective

Verify that EfficientNet-B0 supports the intended training procedure before beginning formal Development training.

### Procedure

1. Load one batch from the Fold 1 training DataLoader.
2. Enable training mode while freezing Batch Normalization running statistics.
3. Execute a forward pass using FP16 automatic mixed precision.
4. Calculate class-weighted binary cross-entropy loss.
5. Perform backward propagation using gradient scaling.
6. Verify that the intended trainable feature blocks and classifier receive finite, nonzero gradients.
7. Confirm that model parameters and Batch Normalization running statistics remain unchanged.

### Safeguards

No optimizer.step() or scaler.step() operation will be performed.

This smoke test does not constitute model training, and its model instance will not be used for formal experiments.

The formal Fold 1 experiment will initialize a fresh ImageNet-pretrained EfficientNet-B0 model.


In [11]:

# ============================================================
# EfficientNet-B0 Gradient Smoke Test
# No optimizer updates
# ============================================================

import torch
import torch.nn as nn
import pandas as pd

from IPython.display import display

assert DEVICE.type == "cuda", (
    "This smoke test expects the configured CUDA runtime."
)

# ------------------------------------------------------------
# 1. Set training mode and freeze BN statistics
# ------------------------------------------------------------

set_efficientnet_training_mode(
    efficientnet_model
)

assert efficientnet_model.training

bn_modules = [
    module
    for module in efficientnet_model.modules()
    if isinstance(module, nn.BatchNorm2d)
]

assert len(bn_modules) == 49

assert all(
    not module.training
    for module in bn_modules
)

# ------------------------------------------------------------
# 2. Save parameter and BN buffer states
# ------------------------------------------------------------

parameter_before = {
    name: parameter.detach().clone()
    for name, parameter
    in efficientnet_model.named_parameters()
}

bn_buffers_before = {
    name: buffer.detach().clone()
    for name, buffer
    in efficientnet_model.named_buffers()
    if any(
        item in name
        for item in [
            "running_mean",
            "running_var",
            "num_batches_tracked"
        ]
    )
}

# ------------------------------------------------------------
# 3. Load one training batch
# ------------------------------------------------------------

batch = next(
    iter(efficientnet_train_loader)
)

images = batch["image"].to(
    DEVICE,
    non_blocking=True
)

labels = batch["label"].to(
    DEVICE,
    dtype=torch.float32,
    non_blocking=True
).reshape(-1, 1)

assert images.shape == (32, 3, 224, 224)
assert labels.shape == (32, 1)

assert torch.isfinite(images).all()
assert torch.isfinite(labels).all()

assert set(
    labels.detach().cpu().flatten().tolist()
).issubset({0.0, 1.0})

# ------------------------------------------------------------
# 4. Forward pass with mixed precision
# ------------------------------------------------------------

optimizer.zero_grad(set_to_none=True)

smoke_scaler = torch.amp.GradScaler("cuda")

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):

    logits = efficientnet_model(images)

    loss = criterion(
        logits,
        labels
    )

assert logits.shape == (32, 1)

assert torch.isfinite(logits).all()
assert torch.isfinite(loss)

loss_value = float(loss.detach().item())

print("=" * 65)
print("EFFICIENTNET-B0 — FORWARD PASS")
print("=" * 65)

print("Input shape:", tuple(images.shape))
print("Logit shape:", tuple(logits.shape))
print("Weighted BCE loss:", round(loss_value, 6))

print("\nPASS: Forward pass and loss verified.")

# ------------------------------------------------------------
# 5. Backward pass without an optimizer step
# ------------------------------------------------------------

smoke_scaler.scale(loss).backward()

# Unscale gradients for accurate inspection.
# This does not update model parameters.
smoke_scaler.unscale_(optimizer)

gradient_rows = []

for group_name, prefix in [
    ("features.7", "features.7."),
    ("features.8", "features.8."),
    ("classifier", "classifier.")
]:

    parameters = [
        parameter
        for name, parameter
        in efficientnet_model.named_parameters()
        if name.startswith(prefix)
        and parameter.requires_grad
    ]

    assert parameters, (
        f"No trainable parameters found: {group_name}"
    )

    gradients = [
        parameter.grad
        for parameter in parameters
    ]

    assert all(
        gradient is not None
        for gradient in gradients
    ), f"Missing gradients in {group_name}"

    assert all(
        torch.isfinite(gradient).all().item()
        for gradient in gradients
    ), f"Non-finite gradients in {group_name}"

    nonzero_gradients = sum(
        int(torch.count_nonzero(gradient).item())
        for gradient in gradients
    )

    assert nonzero_gradients > 0, (
        f"No nonzero gradients in {group_name}"
    )

    gradient_rows.append({
        "group": group_name,
        "parameter_count": sum(
            parameter.numel()
            for parameter in parameters
        ),
        "nonzero_gradient_values": nonzero_gradients,
        "maximum_absolute_gradient": max(
            gradient.abs().max().item()
            for gradient in gradients
        )
    })

gradient_summary = pd.DataFrame(
    gradient_rows
)

print("\n" + "=" * 65)
print("EFFICIENTNET-B0 — GRADIENT VERIFICATION")
print("=" * 65)

display(gradient_summary)

print("\nPASS: Trainable layers received finite gradients.")

# ------------------------------------------------------------
# 6. Verify frozen parameters have no gradients
# ------------------------------------------------------------

frozen_parameters = [
    parameter
    for parameter in efficientnet_model.parameters()
    if not parameter.requires_grad
]

assert all(
    parameter.grad is None
    for parameter in frozen_parameters
)

print("PASS: Frozen parameters received no gradients.")

# ------------------------------------------------------------
# 7. Confirm no weights or BN buffers changed
# ------------------------------------------------------------

for name, parameter in efficientnet_model.named_parameters():

    assert torch.equal(
        parameter.detach(),
        parameter_before[name]
    ), f"Parameter changed unexpectedly: {name}"

for name, buffer in efficientnet_model.named_buffers():

    if name in bn_buffers_before:

        assert torch.equal(
            buffer.detach(),
            bn_buffers_before[name]
        ), f"BatchNorm buffer changed: {name}"

print("PASS: All model parameters unchanged.")
print("PASS: BatchNorm running statistics unchanged.")

# ------------------------------------------------------------
# 8. Clean up smoke-test gradients
# ------------------------------------------------------------

optimizer.zero_grad(set_to_none=True)

del batch, images, labels, logits, loss
del parameter_before, bn_buffers_before
del smoke_scaler

torch.cuda.empty_cache()

# ------------------------------------------------------------
# 9. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("EFFICIENTNET-B0 — GRADIENT SMOKE TEST")
print("=" * 65)

print("PASS: FP16 forward pass successful.")
print("PASS: Weighted loss finite.")
print("PASS: Backward propagation successful.")
print("PASS: All intended trainable groups received gradients.")
print("PASS: Frozen parameters unchanged.")
print("PASS: BatchNorm statistics unchanged.")
print("PASS: No optimizer step performed.")

print("\nReady for fresh-model Fold 1 training.")


EFFICIENTNET-B0 — FORWARD PASS
Input shape: (32, 3, 224, 224)
Logit shape: (32, 1)
Weighted BCE loss: 0.342606

PASS: Forward pass and loss verified.

EFFICIENTNET-B0 — GRADIENT VERIFICATION


,group,parameter_count,nonzero_gradient_values,maximum_absolute_gradient
0,features.7,717232,717232,0.092628
1,features.8,412160,412160,0.005989
2,classifier,1281,1281,0.202271



PASS: Trainable layers received finite gradients.
PASS: Frozen parameters received no gradients.
PASS: All model parameters unchanged.
PASS: BatchNorm running statistics unchanged.

EFFICIENTNET-B0 — GRADIENT SMOKE TEST
PASS: FP16 forward pass successful.
PASS: Weighted loss finite.
PASS: Backward propagation successful.
PASS: All intended trainable groups received gradients.
PASS: Frozen parameters unchanged.
PASS: BatchNorm statistics unchanged.
PASS: No optimizer step performed.

Ready for fresh-model Fold 1 training.



## 5.8 — Formal EfficientNet-B0 Validation Procedure

### Objective

Establish a reproducible validation procedure before beginning formal EfficientNet-B0 training.

### Patch-to-Image Aggregation

Each validation image contributes multiple retained patches.

The primary image-level prediction is obtained by:

1. Collecting the predicted logits of all patches belonging to the same source image.
2. Calculating their arithmetic mean.
3. Applying the sigmoid function to obtain the image-level probability of malignancy.

The alternative mean-probability aggregation is retained in the output for later sensitivity analysis but will not determine checkpoint selection.

### Validation Metrics

- AUROC
- AUPRC
- Balanced accuracy, threshold 0.5
- Expected Calibration Error, 10 equal-width positive-class bins
- Brier score
- Negative log-likelihood, probability clipping at 1e-7

### Checkpoint Selection

After each epoch, calculate validation image-level AUROC.

Retain the checkpoint with the highest AUROC. In the event of an exact tie, retain the earlier checkpoint.

### Quality Control

Before training EfficientNet-B0, independently reproduce the saved ResNet-18 Fold 1 metrics using the same validation formulas.

The validation implementation must preserve patient IDs, image filenames, patch indices, and class labels.

No final Calibration, Test, or external datasets are used.


In [12]:

# ============================================================
# Reusable validation metrics and ResNet-18 audit
# ============================================================

import numpy as np
import pandas as pd
import torch

from scipy.special import expit

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss
)

from IPython.display import display


# ------------------------------------------------------------
# 1. Primary ECE — 10 equal-width positive-class bins
# ------------------------------------------------------------

def compute_primary_ece(y, p, n_bins=10):

    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)

    assert len(y) == len(p)
    assert np.isfinite(p).all()
    assert ((p >= 0) & (p <= 1)).all()

    bin_edges = np.linspace(
        0, 1, n_bins + 1
    )

    bin_ids = np.searchsorted(
        bin_edges[1:-1],
        p,
        side="right"
    )

    ece = 0.0

    for bin_number in range(n_bins):

        mask = bin_ids == bin_number

        if not mask.any():
            continue

        ece += mask.mean() * abs(
            y[mask].mean() - p[mask].mean()
        )

    return float(ece)


# ------------------------------------------------------------
# 2. Image-level performance metrics
# ------------------------------------------------------------

def calculate_efficientnet_metrics(image_df):

    y = image_df["label"].to_numpy(dtype=int)
    p = image_df["probability"].to_numpy(dtype=float)

    assert set(np.unique(y)) == {0, 1}
    assert np.isfinite(p).all()
    assert ((p >= 0) & (p <= 1)).all()

    predicted = (p >= 0.5).astype(int)

    clipped = np.clip(
        p,
        1e-7,
        1.0 - 1e-7
    )

    nll = -np.mean(
        y * np.log(clipped)
        + (1 - y) * np.log1p(-clipped)
    )

    return {
        "auroc": float(roc_auc_score(y, p)),
        "auprc": float(average_precision_score(y, p)),
        "balanced_accuracy": float(
            balanced_accuracy_score(y, predicted)
        ),
        "ece_10": compute_primary_ece(y, p),
        "brier": float(brier_score_loss(y, p)),
        "nll": float(nll)
    }


# ------------------------------------------------------------
# 3. Patch-to-image aggregation
# ------------------------------------------------------------

def aggregate_patch_predictions(patch_df):

    required = {
        "filename",
        "patient_id",
        "label",
        "patch_index",
        "patch_logit"
    }

    assert required.issubset(patch_df.columns)
    assert not patch_df.empty

    assert not patch_df.duplicated(
        ["filename", "patch_index"]
    ).any()

    assert np.isfinite(
        patch_df["patch_logit"].to_numpy(dtype=float)
    ).all()

    data = patch_df.copy()

    data["patient_id"] = (
        data["patient_id"].astype(str)
    )

    data["patch_probability"] = expit(
        data["patch_logit"].to_numpy(dtype=float)
    )

    # Every image must have one patient and one label.
    assert (
        data.groupby("filename")["patient_id"]
        .nunique() == 1
    ).all()

    assert (
        data.groupby("filename")["label"]
        .nunique() == 1
    ).all()

    image_df = (
        data.groupby(
            ["filename", "patient_id", "label"],
            as_index=False
        )
        .agg(
            image_logit=("patch_logit", "mean"),
            mean_patch_probability=(
                "patch_probability", "mean"
            ),
            n_patches=("patch_logit", "size")
        )
    )

    image_df["probability"] = expit(
        image_df["image_logit"].to_numpy(dtype=float)
    )

    image_df["prediction"] = (
        image_df["probability"] >= 0.5
    ).astype(int)

    assert image_df["filename"].is_unique

    assert image_df["n_patches"].sum() == len(
        patch_df
    )

    return image_df


# ------------------------------------------------------------
# 4. EfficientNet-B0 validation inference
# ------------------------------------------------------------

@torch.inference_mode()
def validate_efficientnet(model, loader, device):

    model.eval()

    records = []

    for batch in loader:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(images)

        logits_array = (
            logits.detach()
            .float()
            .reshape(-1)
            .cpu()
            .numpy()
        )

        labels_array = (
            batch["label"]
            .detach()
            .cpu()
            .numpy()
        )

        indices_array = (
            batch["patch_index"]
            .detach()
            .cpu()
            .numpy()
        )

        for i in range(len(logits_array)):

            records.append({
                "filename": str(batch["filename"][i]),
                "patient_id": str(batch["patient_id"][i]),
                "label": int(labels_array[i]),
                "patch_index": int(indices_array[i]),
                "patch_logit": float(logits_array[i])
            })

    patch_df = pd.DataFrame(records)

    image_df = aggregate_patch_predictions(
        patch_df
    )

    metrics = calculate_efficientnet_metrics(
        image_df
    )

    return metrics, patch_df, image_df


# ------------------------------------------------------------
# 5. Audit metric definitions using saved ResNet-18 Fold 1
# ------------------------------------------------------------

reference_path = (
    REPO_ROOT /
    "results" /
    "formal_development" /
    "resnet18" /
    "fold_1" /
    "best_validation_image_predictions.csv"
)

assert reference_path.is_file(), (
    f"Missing reference predictions: {reference_path}"
)

reference_images = pd.read_csv(
    reference_path,
    dtype={"patient_id": str}
)

assert len(reference_images) == 372
assert reference_images["patient_id"].nunique() == 17

recalculated = calculate_efficientnet_metrics(
    reference_images
)

saved_metrics = (
    resnet_comparison.loc[
        resnet_comparison["fold"] == 1
    ].iloc[0]
)

audit_rows = []

for metric, recalculated_value in recalculated.items():

    saved_value = float(saved_metrics[metric])

    difference = abs(
        recalculated_value - saved_value
    )

    audit_rows.append({
        "metric": metric,
        "saved_resnet18": saved_value,
        "recalculated": recalculated_value,
        "absolute_difference": difference
    })

metric_audit = pd.DataFrame(audit_rows)

print("=" * 65)
print("RESNET-18 FOLD 1 — METRIC REPRODUCTION AUDIT")
print("=" * 65)

display(metric_audit.round(8))

assert (
    metric_audit["absolute_difference"] < 1e-6
).all(), "Metric definitions do not reproduce ResNet-18."


# ------------------------------------------------------------
# 6. Verify patch aggregation using saved predictions
# ------------------------------------------------------------

reference_patch_path = (
    REPO_ROOT /
    "results" /
    "formal_development" /
    "resnet18" /
    "fold_1" /
    "best_validation_patch_predictions.csv"
)

assert reference_patch_path.is_file()

reference_patches = pd.read_csv(
    reference_patch_path,
    dtype={"patient_id": str}
)

reconstructed_images = aggregate_patch_predictions(
    reference_patches
)

aggregation_audit = reconstructed_images.merge(
    reference_images[
        ["filename", "probability"]
    ],
    on="filename",
    suffixes=("_reconstructed", "_saved"),
    validate="one_to_one"
)

assert len(aggregation_audit) == 372

maximum_difference = float(
    np.max(np.abs(
        aggregation_audit["probability_reconstructed"]
        - aggregation_audit["probability_saved"]
    ))
)

print("\nMaximum aggregation difference:",
      maximum_difference)

assert maximum_difference < 1e-6

assert len(reference_patches) == 5861

# ------------------------------------------------------------
# 7. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("EFFICIENTNET-B0 — VALIDATION FUNCTION PREFLIGHT")
print("=" * 65)

print("PASS: All six ResNet-18 metrics reproduced.")
print("PASS: Original mean-logit aggregation reproduced.")
print("PASS: Validation function definitions ready.")
print("PASS: No model training performed.")

print("\nReady to initialize a fresh Fold 1 model.")


RESNET-18 FOLD 1 — METRIC REPRODUCTION AUDIT


,metric,saved_resnet18,recalculated,absolute_difference
0,auroc,0.992493,0.992493,0.0
1,auprc,0.996657,0.996657,0.0
2,balanced_accuracy,0.940462,0.940462,0.0
3,ece_10,0.074696,0.074696,0.0
4,brier,0.063208,0.063208,0.0
5,nll,0.256791,0.256791,0.0



Maximum aggregation difference: 1.1102230246251565e-16

EFFICIENTNET-B0 — VALIDATION FUNCTION PREFLIGHT
PASS: All six ResNet-18 metrics reproduced.
PASS: Original mean-logit aggregation reproduced.
PASS: Validation function definitions ready.
PASS: No model training performed.

Ready to initialize a fresh Fold 1 model.



## 5.9 — Formal EfficientNet-B0 Fold 1 Training

### Objective

Train EfficientNet-B0 on Development Folds 2 and 3 and evaluate its performance on the held-out Development Fold 1.

### Training Procedure

1. Initialize a fresh ImageNet-pretrained EfficientNet-B0.
2. Freeze the earlier feature blocks.
3. Fine-tune features[7], features[8], and the binary classifier.
4. Train with AdamW and weighted binary cross-entropy.
5. Use FP16 mixed precision and freeze Batch Normalization running statistics.
6. Complete five training epochs.

### Validation and Checkpoint Selection

After every epoch, calculate image-level validation metrics:

- AUROC
- AUPRC
- Balanced accuracy
- ECE (10 equal-width bins)
- Brier score
- Negative log-likelihood

Select the checkpoint achieving the highest validation image-level AUROC.

If two epochs have exactly equal AUROC, retain the earlier epoch.

The selection criterion will not be changed based on calibration performance.

### Saved Artifacts

Save the following outputs:

- Complete five-epoch training history
- Best validation patch predictions
- Best validation image predictions
- Best validation summary
- AUROC-selected model checkpoint

The checkpoint will be stored outside the GitHub repository because model weights are large binary artifacts.

### Research Safeguards

Only Development data are used.

Calibration, Test, BACH, and BRACS outcomes are excluded.

These results are Development-validation estimates, not independent final Test results.


In [13]:

# ============================================================
# Formal EfficientNet-B0 Fold 1 Training
# ============================================================

import gc
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from tqdm.auto import tqdm
from IPython.display import display

# ------------------------------------------------------------
# 1. Paths and safety checks
# ------------------------------------------------------------

FOLD = 1
SEED = 631
EPOCHS = 5

RESULT_DIR = (
    REPO_ROOT /
    "results" /
    "formal_development" /
    "efficientnet_b0" /
    "fold_1"
)

# Model weights are kept outside GitHub.
CHECKPOINT_DIR = Path(
    "/content/CIS631_model_checkpoints/"
    "efficientnet_b0/fold_1"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

history_path = (
    RESULT_DIR / "training_history.csv"
)

patch_predictions_path = (
    RESULT_DIR /
    "best_validation_patch_predictions.csv"
)

image_predictions_path = (
    RESULT_DIR /
    "best_validation_image_predictions.csv"
)

summary_path = (
    RESULT_DIR /
    "best_validation_summary.json"
)

checkpoint_path = (
    CHECKPOINT_DIR /
    "efficientnet_b0_fold_1_best.pt"
)

# Avoid accidentally overwriting a finished experiment.
assert not (
    summary_path.is_file()
    and pd.read_json(
        summary_path,
        typ="series"
    ).get("training_completed", False)
), (
    "Fold 1 already completed. "
    "Do not overwrite the saved experiment."
)

assert len(train_df) == 12034
assert len(val_df) == 5861

assert train_df["patient_id"].nunique() == 32
assert val_df["patient_id"].nunique() == 17

assert set(train_df["patient_id"]).isdisjoint(
    set(val_df["patient_id"])
)

assert EFFICIENTNET_CONFIG["epochs"] == EPOCHS

assert DEVICE.type == "cuda"

print("PASS: Formal Fold 1 data and configuration verified.")


# ------------------------------------------------------------
# 2. Release smoke-test model
# ------------------------------------------------------------

if "efficientnet_model" in globals():
    del efficientnet_model

if "optimizer" in globals():
    del optimizer

if "criterion" in globals():
    del criterion

gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# 3. Reset random-number generators
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

loader_generator.manual_seed(SEED)

print("PASS: Formal training seed reset.")


# ------------------------------------------------------------
# 4. Initialize fresh EfficientNet-B0
# ------------------------------------------------------------

model = build_efficientnet_b0().to(DEVICE)

total_parameters = sum(
    p.numel() for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

assert total_parameters == 4008829
assert trainable_parameters == 1130673

set_efficientnet_training_mode(model)

assert all(
    not module.training
    for module in model.modules()
    if isinstance(module, nn.BatchNorm2d)
)

print("PASS: Fresh ImageNet-pretrained model initialized.")


# ------------------------------------------------------------
# 5. Fresh optimizer and training-fold loss
# ------------------------------------------------------------

n_benign = int(
    (train_df["label"] == 0).sum()
)

n_malignant = int(
    (train_df["label"] == 1).sum()
)

assert n_benign == 3301
assert n_malignant == 8733

positive_weight = (
    n_benign / n_malignant
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [positive_weight],
        dtype=torch.float32,
        device=DEVICE
    )
)

optimizer = torch.optim.AdamW(
    [
        {
            "params": list(
                model.features[7].parameters()
            ) + list(
                model.features[8].parameters()
            ),
            "lr": 1e-4
        },
        {
            "params": model.classifier.parameters(),
            "lr": 5e-4
        }
    ],
    weight_decay=1e-4
)

scaler = torch.amp.GradScaler("cuda")

optimizer_ids = {
    id(p)
    for group in optimizer.param_groups
    for p in group["params"]
}

trainable_ids = {
    id(p)
    for p in model.parameters()
    if p.requires_grad
}

assert optimizer_ids == trainable_ids

print("PASS: Fresh optimizer and loss initialized.")


# ------------------------------------------------------------
# 6. Define one training epoch
# ------------------------------------------------------------

def train_efficientnet_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
    epoch
):

    # Enables training and then freezes BN statistics.
    set_efficientnet_training_mode(model)

    assert model.training

    assert all(
        not m.training
        for m in model.modules()
        if isinstance(m, nn.BatchNorm2d)
    )

    total_loss = 0.0
    total_examples = 0

    progress = tqdm(
        loader,
        desc=f"Epoch {epoch}/{EPOCHS} training",
        leave=True
    )

    for batch in progress:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            dtype=torch.float32,
            non_blocking=True
        ).reshape(-1, 1)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite loss in epoch {epoch}."
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.shape[0]

        total_loss += (
            float(loss.detach().item())
            * batch_size
        )

        total_examples += batch_size

        progress.set_postfix(
            loss=f"{total_loss / total_examples:.4f}"
        )

    assert total_examples == len(loader.dataset)

    return total_loss / total_examples


# ------------------------------------------------------------
# 7. Formal five-epoch experiment
# ------------------------------------------------------------

best_auc = -float("inf")
best_epoch = None

history_rows = []

print("\n" + "=" * 70)
print("EFFICIENTNET-B0 — FORMAL FOLD 1 TRAINING")
print("=" * 70)

print("Training patients:", 32)
print("Validation patients:", 17)
print("Training patches:", 12034)
print("Validation patches:", 5861)
print("Positive weight:", round(positive_weight, 6))
print("Epochs:", EPOCHS)
print("Device:", DEVICE)
print("=" * 70)

for epoch in range(1, EPOCHS + 1):

    epoch_start = time.perf_counter()

    print(f"\nEpoch {epoch}/{EPOCHS}")

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    training_loss = train_efficientnet_one_epoch(
        model=model,
        loader=efficientnet_train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=DEVICE,
        epoch=epoch
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    print("Running image-level validation...")

    (
        metrics,
        patch_predictions,
        image_predictions
    ) = validate_efficientnet(
        model=model,
        loader=efficientnet_val_loader,
        device=DEVICE
    )

    assert len(patch_predictions) == 5861
    assert len(image_predictions) == 372

    assert (
        image_predictions["patient_id"].nunique()
        == 17
    )

    assert set(
        image_predictions["patient_id"]
    ) == set(val_df["patient_id"])

    assert image_predictions["filename"].is_unique

    assert set(
        pd.MultiIndex.from_frame(
            patch_predictions[
                ["filename", "patch_index"]
            ]
        )
    ) == set(
        pd.MultiIndex.from_frame(
            val_df[
                ["filename", "patch_index"]
            ]
        )
    )

    assert all(
        np.isfinite(value)
        for value in metrics.values()
    )

    # --------------------------------------------------------
    # AUROC-based checkpoint selection
    # --------------------------------------------------------

    selected = (
        metrics["auroc"] > best_auc
    )

    if selected:

        best_auc = float(metrics["auroc"])
        best_epoch = epoch

        # Save original patch and image predictions.
        patch_predictions.to_csv(
            patch_predictions_path,
            index=False
        )

        image_predictions.to_csv(
            image_predictions_path,
            index=False
        )

        # Save weights outside GitHub.
        checkpoint = {
            "architecture": "efficientnet_b0",
            "fold": FOLD,
            "epoch": epoch,
            "selected_validation_auroc": best_auc,
            "training_seed": SEED,
            "model_state_dict": {
                key: tensor.detach().cpu()
                for key, tensor
                in model.state_dict().items()
            },
            "training_config": EFFICIENTNET_CONFIG
        }

        temporary_checkpoint = (
            CHECKPOINT_DIR /
            "efficientnet_b0_fold_1_best.tmp"
        )

        torch.save(
            checkpoint,
            temporary_checkpoint
        )

        temporary_checkpoint.replace(
            checkpoint_path
        )

        print(
            f"New best checkpoint: "
            f"epoch {epoch}, AUROC {best_auc:.6f}"
        )

    # --------------------------------------------------------
    # Save per-epoch results
    # --------------------------------------------------------

    elapsed_seconds = (
        time.perf_counter() - epoch_start
    )

    history_rows.append({
        "epoch": epoch,
        "training_loss": float(training_loss),
        "auroc": float(metrics["auroc"]),
        "auprc": float(metrics["auprc"]),
        "balanced_accuracy": float(
            metrics["balanced_accuracy"]
        ),
        "ece_10": float(metrics["ece_10"]),
        "brier": float(metrics["brier"]),
        "nll": float(metrics["nll"]),
        "selected_new_best": bool(selected),
        "elapsed_seconds": float(elapsed_seconds)
    })

    history_df = pd.DataFrame(history_rows)

    history_df.to_csv(
        history_path,
        index=False
    )

    summary = {
        "architecture": "efficientnet_b0",
        "fold": FOLD,
        "training_patients": 32,
        "validation_patients": 17,
        "training_patches": 12034,
        "validation_patches": 5861,
        "validation_images": 372,
        "epochs_completed": epoch,
        "training_completed": epoch == EPOCHS,
        "selected_epoch": best_epoch,
        "selection_metric": "image_auroc",
        "best_validation_auroc": best_auc,
        "selection_tie_rule": "earliest_epoch",
        "primary_aggregation": "mean_logit",
        "positive_class_weight": float(
            positive_weight
        ),
        "total_parameters": total_parameters,
        "trainable_parameters": trainable_parameters,
        "seed": SEED,
        "selected_validation_metrics": {
            key: float(value)
            for key, value in (
                calculate_efficientnet_metrics(
                    pd.read_csv(image_predictions_path)
                ).items()
            )
        },
        "test_data_used": False,
        "calibration_data_used": False,
        "external_data_used": False
    }

    with open(
        summary_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(summary, f, indent=2)

    print(
        f"Training loss:  {training_loss:.6f}\n"
        f"Image AUROC:    {metrics['auroc']:.6f}\n"
        f"Image AUPRC:    {metrics['auprc']:.6f}\n"
        f"Balanced acc.:  {metrics['balanced_accuracy']:.6f}\n"
        f"ECE (10 bins):  {metrics['ece_10']:.6f}\n"
        f"Brier score:    {metrics['brier']:.6f}\n"
        f"NLL:            {metrics['nll']:.6f}\n"
        f"Epoch seconds:  {elapsed_seconds:.1f}"
    )


# ------------------------------------------------------------
# 8. Verify final experiment artifacts
# ------------------------------------------------------------

assert len(history_rows) == EPOCHS
assert best_epoch in range(1, EPOCHS + 1)

assert checkpoint_path.is_file()
assert history_path.is_file()
assert patch_predictions_path.is_file()
assert image_predictions_path.is_file()
assert summary_path.is_file()

history_df = pd.read_csv(history_path)

best_saved_images = pd.read_csv(
    image_predictions_path,
    dtype={"patient_id": str}
)

best_saved_patches = pd.read_csv(
    patch_predictions_path,
    dtype={"patient_id": str}
)

assert len(best_saved_images) == 372
assert len(best_saved_patches) == 5861

assert best_saved_images["patient_id"].nunique() == 17

assert np.isclose(
    history_df["auroc"].max(),
    best_auc,
    atol=1e-10
)

# First epoch to achieve the maximum AUROC.
assert int(
    history_df.loc[
        history_df["auroc"].idxmax(),
        "epoch"
    ]
) == best_epoch

saved_metrics = calculate_efficientnet_metrics(
    best_saved_images
)

assert np.isclose(
    saved_metrics["auroc"],
    best_auc,
    atol=1e-10
)

# ------------------------------------------------------------
# 9. Display final history
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EFFICIENTNET-B0 — FOLD 1 TRAINING HISTORY")
print("=" * 70)

display(
    history_df[
        [
            "epoch",
            "training_loss",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "ece_10",
            "brier",
            "nll",
            "selected_new_best"
        ]
    ].round(6)
)

print("\nSelected epoch:", best_epoch)
print("Best validation AUROC:", round(best_auc, 6))

print("\nSaved analysis:")
print(RESULT_DIR)

print("\nSaved checkpoint (outside GitHub):")
print(checkpoint_path)

print("\n" + "=" * 70)
print("EFFICIENTNET-B0 — FOLD 1 COMPLETION")
print("=" * 70)

print("PASS: Five training epochs completed.")
print("PASS: Image-level validation completed each epoch.")
print("PASS: Best epoch selected using image AUROC.")
print("PASS: Earlier-epoch AUROC tie rule preserved.")
print("PASS: Training history and predictions saved.")
print("PASS: Best model checkpoint saved outside GitHub.")
print("PASS: Calibration, Test and external data untouched.")


PASS: Formal Fold 1 data and configuration verified.
PASS: Formal training seed reset.
PASS: Fresh ImageNet-pretrained model initialized.
PASS: Fresh optimizer and loss initialized.

EFFICIENTNET-B0 — FORMAL FOLD 1 TRAINING
Training patients: 32
Validation patients: 17
Training patches: 12034
Validation patches: 5861
Positive weight: 0.377992
Epochs: 5
Device: cuda

Epoch 1/5


Epoch 1/5 training:   0%|          | 0/377 [00:00<?, ?it/s]

Running image-level validation...
New best checkpoint: epoch 1, AUROC 0.983791
Training loss:  0.089417
Image AUROC:    0.983791
Image AUPRC:    0.993504
Balanced acc.:  0.936759
ECE (10 bins):  0.084563
Brier score:    0.064297
NLL:            0.223542
Epoch seconds:  342.6

Epoch 2/5


Epoch 2/5 training:   0%|          | 0/377 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f42bbfc8c20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f42bbfc8c20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

...

Running image-level validation...


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f42bbfc8c20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f42bbfc8c20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

New best checkpoint: epoch 5, AUROC 0.988939
Training loss:  0.012422
Image AUROC:    0.988939
Image AUPRC:    0.995159
Balanced acc.:  0.942920
ECE (10 bins):  0.037872
Brier score:    0.043461
NLL:            0.143759
Epoch seconds:  334.5

EFFICIENTNET-B0 — FOLD 1 TRAINING HISTORY


,epoch,training_loss,auroc,auprc,balanced_accuracy,ece_10,brier,nll,selected_new_best
0,1,0.089417,0.983791,0.993504,0.936759,0.084563,0.064297,0.223542,True
1,2,0.032217,0.981300,0.992108,0.932790,0.043602,0.055018,0.200849,False
2,3,0.021931,0.982429,0.992610,0.938719,0.038897,0.049582,0.169977,False
3,4,0.015095,0.988873,0.995342,0.950593,0.062888,0.053062,0.200011,True
4,5,0.012422,0.988939,0.995159,0.942920,0.037872,0.043461,0.143759,True



Selected epoch: 5
Best validation AUROC: 0.988939

Saved analysis:
/content/breast-histology-calibration/results/formal_development/efficientnet_b0/fold_1

Saved checkpoint (outside GitHub):
/content/CIS631_model_checkpoints/efficientnet_b0/fold_1/efficientnet_b0_fold_1_best.pt

EFFICIENTNET-B0 — FOLD 1 COMPLETION
PASS: Five training epochs completed.
PASS: Image-level validation completed each epoch.
PASS: Best epoch selected using image AUROC.
PASS: Earlier-epoch AUROC tie rule preserved.
PASS: Training history and predictions saved.
PASS: Best model checkpoint saved outside GitHub.
PASS: Calibration, Test and external data untouched.



## 5.10 — EfficientNet-B0 Fold 1 Results and Checkpoint Preservation

### Results

The formal EfficientNet-B0 Fold 1 experiment completed five training epochs.

The selected checkpoint was Epoch 5, which achieved the highest validation image-level AUROC of 0.988939.

Selected metrics:

| Metric | Value |
|---|---:|
| AUROC | 0.988939 |
| AUPRC | 0.995159 |
| Balanced accuracy | 0.942920 |
| ECE | 0.037872 |
| Brier score | 0.043461 |
| NLL | 0.143759 |

### Preliminary Architecture Comparison

EfficientNet-B0 produced slightly lower AUROC and AUPRC than ResNet-18 on the same Fold 1 validation cohort, but lower ECE, Brier score, and NLL.

These are Development-stage observations and do not establish overall architectural superiority.

### Checkpoint Preservation

Before starting Fold 2, verify the saved checkpoint, training history, validation predictions, and selected epoch.

Create a SHA-256 integrity record and export a separate backup containing the checkpoint and research results.

Model checkpoints will remain outside GitHub. Research metadata, predictions, and metrics may be version-controlled.

### Next Experiment

Proceed to EfficientNet-B0 Fold 2, training on Development Folds 1 and 3 and validating on Fold 2.


In [14]:

# ============================================================
# Verify and back up EfficientNet-B0 Fold 1
# ============================================================

import hashlib
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from google.colab import files
from IPython.display import display

# ------------------------------------------------------------
# 1. Define saved artifact paths
# ------------------------------------------------------------

fold_dir = (
    REPO_ROOT /
    "results" /
    "formal_development" /
    "efficientnet_b0" /
    "fold_1"
)

checkpoint_file = Path(
    "/content/CIS631_model_checkpoints/"
    "efficientnet_b0/fold_1/"
    "efficientnet_b0_fold_1_best.pt"
)

required_files = [
    fold_dir / "training_history.csv",
    fold_dir / "best_validation_patch_predictions.csv",
    fold_dir / "best_validation_image_predictions.csv",
    fold_dir / "best_validation_summary.json",
    checkpoint_file
]

for path in required_files:
    assert path.is_file(), f"Missing file: {path}"

# ------------------------------------------------------------
# 2. Verify training history
# ------------------------------------------------------------

history = pd.read_csv(
    fold_dir / "training_history.csv"
)

patches = pd.read_csv(
    fold_dir / "best_validation_patch_predictions.csv",
    dtype={"patient_id": str}
)

images = pd.read_csv(
    fold_dir / "best_validation_image_predictions.csv",
    dtype={"patient_id": str}
)

with open(
    fold_dir / "best_validation_summary.json",
    encoding="utf-8"
) as f:
    summary = json.load(f)

assert len(history) == 5
assert history["epoch"].tolist() == [1, 2, 3, 4, 5]

assert len(patches) == 5861
assert len(images) == 372
assert images["patient_id"].nunique() == 17
assert images["filename"].is_unique

assert not patches.duplicated(
    ["filename", "patch_index"]
).any()

assert summary["training_completed"] is True
assert summary["epochs_completed"] == 5
assert summary["selected_epoch"] == 5

selected_epoch = int(
    history.loc[
        history["auroc"].idxmax(),
        "epoch"
    ]
)

assert selected_epoch == 5

saved_metrics = calculate_efficientnet_metrics(images)

assert np.isclose(
    saved_metrics["auroc"],
    history["auroc"].max(),
    atol=1e-10
)

for metric, value in saved_metrics.items():

    assert np.isclose(
        value,
        summary["selected_validation_metrics"][metric],
        atol=1e-6
    )

print("PASS: Five-epoch history verified.")
print("PASS: Selected epoch and metrics verified.")
print("PASS: Validation image and patch files verified.")

# ------------------------------------------------------------
# 3. Verify checkpoint can be loaded
# ------------------------------------------------------------

checkpoint = torch.load(
    checkpoint_file,
    map_location="cpu",
    weights_only=True
)

assert checkpoint["architecture"] == "efficientnet_b0"
assert checkpoint["fold"] == 1
assert checkpoint["epoch"] == 5

assert np.isclose(
    checkpoint["selected_validation_auroc"],
    saved_metrics["auroc"],
    atol=1e-10
)

# Verify checkpoint keys and tensor dimensions
verification_model = build_efficientnet_b0()

verification_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

assert all(
    torch.isfinite(tensor).all()
    for tensor in verification_model.state_dict().values()
    if tensor.is_floating_point()
)

del verification_model, checkpoint

print("PASS: Checkpoint loaded successfully.")
print("PASS: All model parameters and buffers verified.")

# ------------------------------------------------------------
# 4. Generate checkpoint SHA-256
# ------------------------------------------------------------

sha256 = hashlib.sha256()

with open(checkpoint_file, "rb") as f:

    for chunk in iter(
        lambda: f.read(1024 * 1024),
        b""
    ):
        sha256.update(chunk)

checkpoint_hash = sha256.hexdigest()

integrity_record = {
    "architecture": "efficientnet_b0",
    "fold": 1,
    "selected_epoch": 5,
    "validation_images": 372,
    "validation_patients": 17,
    "validation_patches": 5861,
    "best_validation_auroc": float(
        saved_metrics["auroc"]
    ),
    "checkpoint_filename": checkpoint_file.name,
    "checkpoint_sha256": checkpoint_hash,
    "checkpoint_size_bytes": checkpoint_file.stat().st_size,
    "training_completed": True,
    "test_data_used": False,
    "calibration_data_used": False
}

integrity_path = (
    fold_dir / "checkpoint_integrity.json"
)

with open(integrity_path, "w") as f:
    json.dump(integrity_record, f, indent=2)

print("\nCheckpoint SHA-256:")
print(checkpoint_hash)

# ------------------------------------------------------------
# 5. Create external backup ZIP
# ------------------------------------------------------------

backup_zip = Path(
    "/content/CIS631_EfficientNetB0_Fold1_Backup.zip"
)

with zipfile.ZipFile(
    backup_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for path in sorted(fold_dir.iterdir()):

        if path.is_file() and path.suffix.lower() in {
            ".csv", ".json"
        }:

            zf.write(
                path,
                arcname=str(
                    path.relative_to(REPO_ROOT)
                )
            )

    # Keep checkpoint in a clearly separate folder
    zf.write(
        checkpoint_file,
        arcname=(
            "checkpoints/efficientnet_b0/fold_1/"
            + checkpoint_file.name
        )
    )

with zipfile.ZipFile(backup_zip) as zf:

    assert zf.testzip() is None

    assert (
        "checkpoints/efficientnet_b0/fold_1/"
        + checkpoint_file.name
    ) in zf.namelist()

# ------------------------------------------------------------
# 6. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("EFFICIENTNET-B0 — FOLD 1 BACKUP")
print("=" * 65)

print("Selected epoch:", selected_epoch)
print("Best AUROC:", round(saved_metrics["auroc"], 6))

print("Checkpoint size (MB):",
      round(checkpoint_file.stat().st_size / 1024**2, 2))

print("Backup ZIP:", backup_zip)

print("Backup ZIP size (MB):",
      round(backup_zip.stat().st_size / 1024**2, 2))

print("\nPASS: Fold 1 results verified.")
print("PASS: Checkpoint integrity recorded.")
print("PASS: Backup ZIP created and verified.")

# Download backup to local computer
files.download(str(backup_zip))


PASS: Five-epoch history verified.
PASS: Selected epoch and metrics verified.
PASS: Validation image and patch files verified.
PASS: Checkpoint loaded successfully.
PASS: All model parameters and buffers verified.

Checkpoint SHA-256:
387faf450467c89809df1c5f23d039d4ea5f0a77536f420d398c6013e035dd34

EFFICIENTNET-B0 — FOLD 1 BACKUP
Selected epoch: 5
Best AUROC: 0.988939
Checkpoint size (MB): 15.57
Backup ZIP: /content/CIS631_EfficientNetB0_Fold1_Backup.zip
Backup ZIP size (MB): 14.42

PASS: Fold 1 results verified.
PASS: Checkpoint integrity recorded.
PASS: Backup ZIP created and verified.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


## 5.11 — EfficientNet-B0 Fold 2 Data Preparation

### Objective

Prepare the second patient-grouped Development fold for formal EfficientNet-B0 training.

### Dataset Partition

- Training: Development Folds 1 and 3
- Validation: Development Fold 2
- Training patients: 33
- Validation patients: 16
- Training patches: 11,085
- Validation patches: 6,810

### Protocol Consistency

Reuse the same preprocessing functions, augmentation, normalization, model architecture, loss formulation, optimizer settings, and validation metrics as Fold 1.

Calculate the class weight using Fold 2 training data only.

A fresh ImageNet-pretrained EfficientNet-B0 will be initialized for Fold 2. The Fold 1 trained checkpoint must not be used to initialize the new experiment.

No Calibration, Test, or external data are accessed.

### Validation

Before training, verify the patient separation, sample counts, class labels, dataset loading, and DataLoader configuration.


In [15]:

# ============================================================
# EfficientNet-B0 Fold 2 Data Preparation
# ============================================================

import math
import torch
import pandas as pd

from torch.utils.data import DataLoader
from IPython.display import display

# ------------------------------------------------------------
# 1. Create Fold 2 partitions
# ------------------------------------------------------------

FOLD_2 = 2

fold2_train_df = manifest.loc[
    manifest["development_fold"] != FOLD_2
].copy().reset_index(drop=True)

fold2_val_df = manifest.loc[
    manifest["development_fold"] == FOLD_2
].copy().reset_index(drop=True)

train_patients = set(fold2_train_df["patient_id"])
val_patients = set(fold2_val_df["patient_id"])

assert train_patients.isdisjoint(val_patients)

assert len(train_patients) == 33
assert len(val_patients) == 16

assert len(fold2_train_df) == 11085
assert len(fold2_val_df) == 6810

assert fold2_train_df["filename"].nunique() == 702
assert fold2_val_df["filename"].nunique() == 431

assert set(fold2_train_df["label"].unique()) == {0, 1}
assert set(fold2_val_df["label"].unique()) == {0, 1}

# Confirm that every Development patient belongs
# to exactly one of these partitions.
assert (
    train_patients | val_patients
) == set(manifest["patient_id"])

# ------------------------------------------------------------
# 2. Reuse original patch Dataset
# ------------------------------------------------------------

fold2_train_dataset = BreakHisPatchDataset(
    manifest=fold2_train_df,
    dataset_root=DATASET_ROOT,
    transform=efficientnet_train_transform
)

fold2_val_dataset = BreakHisPatchDataset(
    manifest=fold2_val_df,
    dataset_root=DATASET_ROOT,
    transform=efficientnet_val_transform
)

assert len(fold2_train_dataset) == 11085
assert len(fold2_val_dataset) == 6810

# ------------------------------------------------------------
# 3. Verify source patch loading
# ------------------------------------------------------------

train_sample = fold2_train_dataset[0]
val_sample = fold2_val_dataset[0]

train_image = get_image_tensor(train_sample)
val_image = get_image_tensor(val_sample)

assert train_image.shape == (3, 224, 224)
assert val_image.shape == (3, 224, 224)

assert torch.isfinite(train_image).all()
assert torch.isfinite(val_image).all()

val_image_repeat = get_image_tensor(
    fold2_val_dataset[0]
)

assert torch.equal(val_image, val_image_repeat)

# ------------------------------------------------------------
# 4. Create separate Fold 2 DataLoaders
# ------------------------------------------------------------

fold2_generator = torch.Generator()
fold2_generator.manual_seed(631)

fold2_train_loader = DataLoader(
    fold2_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    generator=fold2_generator
)

fold2_val_loader = DataLoader(
    fold2_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

assert len(fold2_train_loader) == math.ceil(11085 / 32)
assert len(fold2_val_loader) == math.ceil(6810 / 32)

# ------------------------------------------------------------
# 5. Compute training-only class weight
# ------------------------------------------------------------

fold2_benign = int(
    (fold2_train_df["label"] == 0).sum()
)

fold2_malignant = int(
    (fold2_train_df["label"] == 1).sum()
)

assert fold2_benign > 0
assert fold2_malignant > 0

fold2_pos_weight = (
    fold2_benign / fold2_malignant
)

# ------------------------------------------------------------
# 6. Display Fold 2 summary
# ------------------------------------------------------------

fold2_summary = pd.DataFrame([
    {
        "partition": "Training",
        "patients": len(train_patients),
        "images": fold2_train_df["filename"].nunique(),
        "patches": len(fold2_train_df)
    },
    {
        "partition": "Validation",
        "patients": len(val_patients),
        "images": fold2_val_df["filename"].nunique(),
        "patches": len(fold2_val_df)
    }
])

print("=" * 65)
print("EFFICIENTNET-B0 — FOLD 2 DATA PREPARATION")
print("=" * 65)

display(fold2_summary)

print("\nTraining batches:", len(fold2_train_loader))
print("Validation batches:", len(fold2_val_loader))

print("\nBenign training patches:", fold2_benign)
print("Malignant training patches:", fold2_malignant)
print("Positive-class weight:", round(fold2_pos_weight, 6))

print("\nPASS: Fold 2 patient separation verified.")
print("PASS: Frozen image and patch counts verified.")
print("PASS: Original preprocessing reused.")
print("PASS: Fold 2 datasets load successfully.")
print("PASS: DataLoaders configured.")
print("PASS: Class weight calculated from training only.")
print("PASS: No Fold 2 model training performed.")


EFFICIENTNET-B0 — FOLD 2 DATA PREPARATION


,partition,patients,images,patches
0,Training,33,702,11085
1,Validation,16,431,6810



Training batches: 347
Validation batches: 213

Benign training patches: 3109
Malignant training patches: 7976
Positive-class weight: 0.389794

PASS: Fold 2 patient separation verified.
PASS: Frozen image and patch counts verified.
PASS: Original preprocessing reused.
PASS: Fold 2 datasets load successfully.
PASS: DataLoaders configured.
PASS: Class weight calculated from training only.
PASS: No Fold 2 model training performed.



## 5.12 — Formal EfficientNet-B0 Fold 2 Training

### Objective

Train EfficientNet-B0 using Development Folds 1 and 3, with Development Fold 2 reserved for validation.

### Training Protocol

The procedure remains identical to Fold 1:

- Fresh ImageNet-pretrained EfficientNet-B0
- Fine-tuning of features[7], features[8], and classifier
- Batch size 32
- Five training epochs
- AdamW optimizer with learning rates 0.0001 and 0.0005
- Weight decay 0.0001
- Fold-specific class-weighted binary cross-entropy
- FP16 mixed precision with frozen Batch Normalization statistics
- Random seed 631

### Validation

After each epoch, aggregate patch logits into image-level malignancy probabilities and calculate AUROC, AUPRC, balanced accuracy, ECE, Brier score, and NLL.

Select the checkpoint with the highest validation image-level AUROC, retaining the earlier epoch in the event of a tie.

### Outputs

Save the five-epoch history, selected patch predictions, selected image predictions, summary metadata, and best model checkpoint.

Fold 2 artifacts will be stored separately from Fold 1 artifacts. The model checkpoint will remain outside GitHub.

Calibration, Test, and external evaluation datasets are excluded.


In [16]:

# ============================================================
# Formal EfficientNet-B0 Fold 2 Training
# ============================================================

import gc
import json
import random
import time

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from IPython.display import display

# ------------------------------------------------------------
# 1. Verify protocol and safeguard Fold 1
# ------------------------------------------------------------

FOLD_2 = 2
SEED = 631
EPOCHS = 5

assert DEVICE.type == "cuda"

assert callable(build_efficientnet_b0)
assert callable(train_efficientnet_one_epoch)
assert callable(validate_efficientnet)

assert EFFICIENTNET_CONFIG["epochs"] == 5
assert EFFICIENTNET_CONFIG["batch_size"] == 32

assert set(fold2_train_df["development_fold"]) == {1, 3}
assert set(fold2_val_df["development_fold"]) == {2}

assert len(fold2_train_df) == 11085
assert len(fold2_val_df) == 6810

assert fold2_train_df["patient_id"].nunique() == 33
assert fold2_val_df["patient_id"].nunique() == 16

assert set(fold2_train_df["patient_id"]).isdisjoint(
    set(fold2_val_df["patient_id"])
)

fold1_checkpoint = Path(
    "/content/CIS631_model_checkpoints/"
    "efficientnet_b0/fold_1/"
    "efficientnet_b0_fold_1_best.pt"
)

assert fold1_checkpoint.is_file(), (
    "Fold 1 checkpoint was not found."
)

fold1_summary = (
    REPO_ROOT /
    "results/formal_development/efficientnet_b0/"
    "fold_1/best_validation_summary.json"
)

assert fold1_summary.is_file()

with open(fold1_summary, encoding="utf-8") as f:
    previous_summary = json.load(f)

assert previous_summary["training_completed"] is True
assert previous_summary["selected_epoch"] == 5

print("PASS: Fold 1 checkpoint and results protected.")
print("PASS: Fold 2 patient separation verified.")

# ------------------------------------------------------------
# 2. Prepare separate Fold 2 output paths
# ------------------------------------------------------------

fold2_result_dir = (
    REPO_ROOT /
    "results/formal_development/efficientnet_b0/fold_2"
)

fold2_checkpoint_dir = Path(
    "/content/CIS631_model_checkpoints/"
    "efficientnet_b0/fold_2"
)

fold2_history_path = (
    fold2_result_dir / "training_history.csv"
)

fold2_patch_path = (
    fold2_result_dir /
    "best_validation_patch_predictions.csv"
)

fold2_image_path = (
    fold2_result_dir /
    "best_validation_image_predictions.csv"
)

fold2_summary_path = (
    fold2_result_dir /
    "best_validation_summary.json"
)

fold2_checkpoint_path = (
    fold2_checkpoint_dir /
    "efficientnet_b0_fold_2_best.pt"
)

artifact_paths = [
    fold2_history_path,
    fold2_patch_path,
    fold2_image_path,
    fold2_summary_path,
    fold2_checkpoint_path
]

assert not any(path.exists() for path in artifact_paths), (
    "Fold 2 output files already exist. "
    "Inspect them before rerunning this training cell."
)

fold2_result_dir.mkdir(parents=True, exist_ok=True)
fold2_checkpoint_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. Release previous model and optimizer from memory
# ------------------------------------------------------------

for variable_name in (
    "model",
    "optimizer",
    "criterion",
    "scaler"
):
    globals().pop(variable_name, None)

gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 4. Reset random seeds
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

fold2_generator.manual_seed(SEED)

print("PASS: Formal Fold 2 random seed reset.")

# ------------------------------------------------------------
# 5. Initialize a fresh ImageNet-pretrained model
# ------------------------------------------------------------

model = build_efficientnet_b0().to(DEVICE)

total_parameters = sum(
    p.numel() for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

assert total_parameters == 4008829
assert trainable_parameters == 1130673

set_efficientnet_training_mode(model)

assert all(
    not module.training
    for module in model.modules()
    if isinstance(module, nn.BatchNorm2d)
)

print("PASS: Fresh EfficientNet-B0 model initialized.")

# ------------------------------------------------------------
# 6. Configure Fold 2 class-weighted loss
# ------------------------------------------------------------

n_benign = int(
    (fold2_train_df["label"] == 0).sum()
)

n_malignant = int(
    (fold2_train_df["label"] == 1).sum()
)

assert n_benign == 3109
assert n_malignant == 7976

positive_weight = n_benign / n_malignant

assert np.isclose(
    positive_weight,
    fold2_pos_weight
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [positive_weight],
        device=DEVICE,
        dtype=torch.float32
    )
)

# ------------------------------------------------------------
# 7. Fresh AdamW optimizer and GradScaler
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    [
        {
            "params": (
                list(model.features[7].parameters())
                + list(model.features[8].parameters())
            ),
            "lr": 1e-4
        },
        {
            "params": model.classifier.parameters(),
            "lr": 5e-4
        }
    ],
    weight_decay=1e-4
)

scaler = torch.amp.GradScaler("cuda")

optimizer_param_ids = [
    id(p)
    for group in optimizer.param_groups
    for p in group["params"]
]

trainable_param_ids = [
    id(p)
    for p in model.parameters()
    if p.requires_grad
]

assert len(optimizer_param_ids) == len(
    set(optimizer_param_ids)
)

assert set(optimizer_param_ids) == set(
    trainable_param_ids
)

print("PASS: Fold-specific loss and optimizer verified.")

# ------------------------------------------------------------
# 8. Begin formal five-epoch training
# ------------------------------------------------------------

best_auc = -float("inf")
best_epoch = None
best_metrics = None

history_rows = []

print("\n" + "=" * 70)
print("EFFICIENTNET-B0 — FORMAL FOLD 2 TRAINING")
print("=" * 70)

print("Training patients:", 33)
print("Validation patients:", 16)
print("Training patches:", 11085)
print("Validation patches:", 6810)
print("Training batches:", len(fold2_train_loader))
print("Validation batches:", len(fold2_val_loader))
print("Positive weight:", round(positive_weight, 6))
print("Epochs:", EPOCHS)
print("Device:", DEVICE)
print("=" * 70)

for epoch in range(1, EPOCHS + 1):

    epoch_start = time.perf_counter()

    print(f"\nEpoch {epoch}/{EPOCHS}")

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    training_loss = train_efficientnet_one_epoch(
        model=model,
        loader=fold2_train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=DEVICE,
        epoch=epoch
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    print("Running image-level validation...")

    metrics, patch_predictions, image_predictions = (
        validate_efficientnet(
            model=model,
            loader=fold2_val_loader,
            device=DEVICE
        )
    )

    assert len(patch_predictions) == 6810
    assert len(image_predictions) == 431

    assert image_predictions["patient_id"].nunique() == 16
    assert image_predictions["filename"].is_unique

    assert set(image_predictions["patient_id"]) == set(
        fold2_val_df["patient_id"].astype(str)
    )

    expected_patch_ids = pd.MultiIndex.from_frame(
        fold2_val_df[["filename", "patch_index"]]
    )

    predicted_patch_ids = pd.MultiIndex.from_frame(
        patch_predictions[["filename", "patch_index"]]
    )

    assert predicted_patch_ids.is_unique

    assert set(predicted_patch_ids) == set(
        expected_patch_ids
    )

    assert all(
        np.isfinite(value)
        for value in metrics.values()
    )

    # --------------------------------------------------------
    # Select checkpoint using image-level AUROC only
    # --------------------------------------------------------

    selected = metrics["auroc"] > best_auc

    if selected:

        best_auc = float(metrics["auroc"])
        best_epoch = epoch

        best_metrics = {
            key: float(value)
            for key, value in metrics.items()
        }

        # Save selected validation predictions
        patch_predictions.to_csv(
            fold2_patch_path,
            index=False
        )

        image_predictions.to_csv(
            fold2_image_path,
            index=False
        )

        # Save selected model weights outside GitHub
        checkpoint = {
            "architecture": "efficientnet_b0",
            "fold": FOLD_2,
            "epoch": epoch,
            "selected_validation_auroc": best_auc,
            "training_seed": SEED,
            "model_state_dict": {
                name: tensor.detach().cpu()
                for name, tensor
                in model.state_dict().items()
            },
            "training_config": EFFICIENTNET_CONFIG
        }

        temporary_path = (
            fold2_checkpoint_dir /
            "efficientnet_b0_fold_2_best.tmp"
        )

        torch.save(checkpoint, temporary_path)

        temporary_path.replace(
            fold2_checkpoint_path
        )

        print(
            f"New best checkpoint: epoch {epoch}, "
            f"AUROC {best_auc:.6f}"
        )

    # --------------------------------------------------------
    # Save epoch history and selected-model summary
    # --------------------------------------------------------

    elapsed_seconds = (
        time.perf_counter() - epoch_start
    )

    history_rows.append({
        "epoch": epoch,
        "training_loss": float(training_loss),
        "auroc": float(metrics["auroc"]),
        "auprc": float(metrics["auprc"]),
        "balanced_accuracy": float(
            metrics["balanced_accuracy"]
        ),
        "ece_10": float(metrics["ece_10"]),
        "brier": float(metrics["brier"]),
        "nll": float(metrics["nll"]),
        "selected_new_best": bool(selected),
        "elapsed_seconds": float(elapsed_seconds)
    })

    pd.DataFrame(history_rows).to_csv(
        fold2_history_path,
        index=False
    )

    summary = {
        "architecture": "efficientnet_b0",
        "fold": FOLD_2,
        "training_patients": 33,
        "validation_patients": 16,
        "training_patches": 11085,
        "validation_patches": 6810,
        "validation_images": 431,
        "epochs_completed": epoch,
        "training_completed": epoch == EPOCHS,
        "selected_epoch": best_epoch,
        "selection_metric": "image_auroc",
        "best_validation_auroc": best_auc,
        "selection_tie_rule": "earliest_epoch",
        "primary_aggregation": "mean_logit",
        "positive_class_weight": float(positive_weight),
        "total_parameters": total_parameters,
        "trainable_parameters": trainable_parameters,
        "seed": SEED,
        "selected_validation_metrics": best_metrics,
        "calibration_data_used": False,
        "test_data_used": False,
        "external_data_used": False
    }

    with open(
        fold2_summary_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(summary, f, indent=2)

    print(
        f"Training loss:  {training_loss:.6f}\n"
        f"Image AUROC:    {metrics['auroc']:.6f}\n"
        f"Image AUPRC:    {metrics['auprc']:.6f}\n"
        f"Balanced acc.:  {metrics['balanced_accuracy']:.6f}\n"
        f"ECE (10 bins):  {metrics['ece_10']:.6f}\n"
        f"Brier score:    {metrics['brier']:.6f}\n"
        f"NLL:            {metrics['nll']:.6f}\n"
        f"Epoch seconds:  {elapsed_seconds:.1f}"
    )

# ------------------------------------------------------------
# 9. Verify completed Fold 2 experiment
# ------------------------------------------------------------

history_df = pd.read_csv(
    fold2_history_path
)

best_images = pd.read_csv(
    fold2_image_path,
    dtype={"patient_id": str}
)

best_patches = pd.read_csv(
    fold2_patch_path,
    dtype={"patient_id": str}
)

with open(
    fold2_summary_path,
    encoding="utf-8"
) as f:
    final_summary = json.load(f)

assert len(history_df) == 5
assert history_df["epoch"].tolist() == [1, 2, 3, 4, 5]

assert len(best_images) == 431
assert len(best_patches) == 6810

assert best_images["patient_id"].nunique() == 16

assert best_epoch == int(
    history_df.loc[
        history_df["auroc"].idxmax(),
        "epoch"
    ]
)

assert np.isclose(
    best_auc,
    history_df["auroc"].max(),
    atol=1e-10
)

recalculated = calculate_efficientnet_metrics(
    best_images
)

for metric, value in recalculated.items():

    assert np.isclose(
        value,
        best_metrics[metric],
        atol=1e-6
    )

assert final_summary["training_completed"] is True
assert final_summary["selected_epoch"] == best_epoch

assert fold2_checkpoint_path.is_file()

# ------------------------------------------------------------
# 10. Display final results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EFFICIENTNET-B0 — FOLD 2 TRAINING HISTORY")
print("=" * 70)

display(
    history_df[
        [
            "epoch",
            "training_loss",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "ece_10",
            "brier",
            "nll",
            "selected_new_best"
        ]
    ].round(6)
)

print("\nSelected epoch:", best_epoch)
print("Best validation AUROC:", round(best_auc, 6))

print("\nResults:")
print(fold2_result_dir)

print("\nCheckpoint (outside GitHub):")
print(fold2_checkpoint_path)

print("\n" + "=" * 70)
print("EFFICIENTNET-B0 — FOLD 2 COMPLETION")
print("=" * 70)

print("PASS: Five training epochs completed.")
print("PASS: Image-level validation completed each epoch.")
print("PASS: Highest-AUROC checkpoint selected.")
print("PASS: Earlier-epoch tie rule preserved.")
print("PASS: Fold 2 history and predictions saved.")
print("PASS: Fold 2 checkpoint saved outside GitHub.")
print("PASS: Fold 1 results preserved.")
print("PASS: Calibration, Test and external data untouched.")


PASS: Fold 1 checkpoint and results protected.
PASS: Fold 2 patient separation verified.
PASS: Formal Fold 2 random seed reset.
PASS: Fresh EfficientNet-B0 model initialized.
PASS: Fold-specific loss and optimizer verified.

EFFICIENTNET-B0 — FORMAL FOLD 2 TRAINING
Training patients: 33
Validation patients: 16
Training patches: 11085
Validation patches: 6810
Training batches: 347
Validation batches: 213
Positive weight: 0.389794
Epochs: 5
Device: cuda

Epoch 1/5


Epoch 1/5 training:   0%|          | 0/347 [00:00<?, ?it/s]

Running image-level validation...


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f42bbfc8c20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f42bbfc8c20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

New best checkpoint: epoch 1, AUROC 0.959328
Training loss:  0.080225
Image AUROC:    0.959328
Image AUPRC:    0.984838
Balanced acc.:  0.869973
ECE (10 bins):  0.083533
Brier score:    0.085128
NLL:            0.297033
Epoch seconds:  331.7

Epoch 2/5


...

Running image-level validation...


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f42bbfc8c20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    
if w.is_alive():  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f42bbfc8c20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

New best checkpoint: epoch 5, AUROC 0.969420
Training loss:  0.007796
Image AUROC:    0.969420
Image AUPRC:    0.988223
Balanced acc.:  0.848691
ECE (10 bins):  0.085673
Brier score:    0.088800
NLL:            0.369099
Epoch seconds:  337.1

EFFICIENTNET-B0 — FOLD 2 TRAINING HISTORY


,epoch,training_loss,auroc,auprc,balanced_accuracy,ece_10,brier,nll,selected_new_best
0,1,0.080225,0.959328,0.984838,0.869973,0.083533,0.085128,0.297033,True
1,2,0.027222,0.965156,0.986541,0.895014,0.063011,0.073279,0.294562,True
2,3,0.016702,0.965333,0.986763,0.907567,0.076435,0.070203,0.317404,True
3,4,0.011082,0.967124,0.987149,0.864132,0.072972,0.082492,0.321350,True
4,5,0.007796,0.969420,0.988223,0.848691,0.085673,0.088800,0.369099,True



Selected epoch: 5
Best validation AUROC: 0.96942

Results:
/content/breast-histology-calibration/results/formal_development/efficientnet_b0/fold_2

Checkpoint (outside GitHub):
/content/CIS631_model_checkpoints/efficientnet_b0/fold_2/efficientnet_b0_fold_2_best.pt

EFFICIENTNET-B0 — FOLD 2 COMPLETION
PASS: Five training epochs completed.
PASS: Image-level validation completed each epoch.
PASS: Highest-AUROC checkpoint selected.
PASS: Earlier-epoch tie rule preserved.
PASS: Fold 2 history and predictions saved.
PASS: Fold 2 checkpoint saved outside GitHub.
PASS: Fold 1 results preserved.
PASS: Calibration, Test and external data untouched.



## 5.13 — EfficientNet-B0 Fold 2 Verification and Backup

### Selected Checkpoint

The formal Fold 2 experiment completed all five epochs.

Epoch 5 achieved the highest validation image-level AUROC (0.969420) and was selected according to the predefined checkpoint-selection protocol.

### Preliminary Findings

EfficientNet-B0 improved all six selected-checkpoint validation metrics relative to ResNet-18 on Fold 2.

However, validation Brier score and NLL increased during later epochs even as training loss decreased and AUROC improved. This highlights the distinction between discrimination and probability quality.

These findings remain preliminary Development results, not independent Test results.

### Reproducibility and Preservation

Before beginning Fold 3:

1. Verify the five-epoch training history.
2. Recalculate the selected image-level metrics.
3. Reconstruct image probabilities from saved patch logits.
4. Verify that the checkpoint loads successfully.
5. Generate SHA-256 integrity records.
6. Export a ZIP backup containing Fold 2 results and checkpoint weights.

The model checkpoint remains outside GitHub. The notebook, predictions, metrics, and integrity records are suitable for version control.


In [17]:

# ============================================================
# Verify and Back Up EfficientNet-B0 Fold 2
# ============================================================

import gc
import hashlib
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from google.colab import files
from IPython.display import display

# ------------------------------------------------------------
# 1. Locate Fold 2 artifacts
# ------------------------------------------------------------

fold_dir = (
    REPO_ROOT /
    "results/formal_development/efficientnet_b0/fold_2"
)

checkpoint_file = Path(
    "/content/CIS631_model_checkpoints/"
    "efficientnet_b0/fold_2/"
    "efficientnet_b0_fold_2_best.pt"
)

history_file = fold_dir / "training_history.csv"

patch_file = (
    fold_dir / "best_validation_patch_predictions.csv"
)

image_file = (
    fold_dir / "best_validation_image_predictions.csv"
)

summary_file = (
    fold_dir / "best_validation_summary.json"
)

required_files = [
    history_file,
    patch_file,
    image_file,
    summary_file,
    checkpoint_file
]

for path in required_files:
    assert path.is_file(), f"Missing artifact: {path}"

print("PASS: All Fold 2 artifacts located.")

# ------------------------------------------------------------
# 2. Verify training history and selection
# ------------------------------------------------------------

history = pd.read_csv(history_file)

patches = pd.read_csv(
    patch_file,
    dtype={"patient_id": str}
)

images = pd.read_csv(
    image_file,
    dtype={"patient_id": str}
)

with open(summary_file, encoding="utf-8") as f:
    summary = json.load(f)

assert history["epoch"].tolist() == [1, 2, 3, 4, 5]

assert len(patches) == 6810
assert len(images) == 431

assert images["patient_id"].nunique() == 16
assert images["filename"].is_unique

assert not patches.duplicated(
    ["filename", "patch_index"]
).any()

assert summary["training_completed"] is True
assert summary["epochs_completed"] == 5
assert summary["selected_epoch"] == 5

selected_epoch = int(
    history.loc[
        history["auroc"].idxmax(), "epoch"
    ]
)

assert selected_epoch == 5

print("PASS: Training history and selection verified.")

# ------------------------------------------------------------
# 3. Recalculate selected validation metrics
# ------------------------------------------------------------

metrics = calculate_efficientnet_metrics(images)

for metric, value in metrics.items():
    assert np.isclose(
        value,
        summary["selected_validation_metrics"][metric],
        atol=1e-6
    )

assert np.isclose(
    metrics["auroc"],
    history["auroc"].max(),
    atol=1e-10
)

print("PASS: All six validation metrics reproduced.")

# ------------------------------------------------------------
# 4. Reconstruct image probabilities from patches
# ------------------------------------------------------------

reconstructed = aggregate_patch_predictions(patches)

comparison = reconstructed[
    ["filename", "probability"]
].merge(
    images[["filename", "probability"]],
    on="filename",
    suffixes=("_reconstructed", "_saved"),
    validate="one_to_one"
)

assert len(comparison) == 431

max_difference = float(
    np.max(
        np.abs(
            comparison["probability_reconstructed"]
            - comparison["probability_saved"]
        )
    )
)

assert max_difference < 1e-6

print(
    "Maximum aggregation difference:",
    max_difference
)

print("PASS: Mean-logit aggregation verified.")

# ------------------------------------------------------------
# 5. Verify checkpoint weights
# ------------------------------------------------------------

checkpoint = torch.load(
    checkpoint_file,
    map_location="cpu",
    weights_only=True
)

assert checkpoint["architecture"] == "efficientnet_b0"
assert checkpoint["fold"] == 2
assert checkpoint["epoch"] == 5

assert np.isclose(
    checkpoint["selected_validation_auroc"],
    metrics["auroc"],
    atol=1e-10
)

verification_model = build_efficientnet_b0()

verification_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

assert all(
    torch.isfinite(t).all()
    for t in verification_model.state_dict().values()
    if t.is_floating_point()
)

del verification_model, checkpoint
gc.collect()

print("PASS: Saved checkpoint loads correctly.")

# ------------------------------------------------------------
# 6. SHA-256 integrity records
# ------------------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


checkpoint_hash = sha256_file(checkpoint_file)

integrity_record = {
    "architecture": "efficientnet_b0",
    "fold": 2,
    "selected_epoch": selected_epoch,
    "best_validation_auroc": metrics["auroc"],
    "validation_patients": 16,
    "validation_images": 431,
    "validation_patches": 6810,
    "checkpoint_filename": checkpoint_file.name,
    "checkpoint_sha256": checkpoint_hash,
    "checkpoint_size_bytes": checkpoint_file.stat().st_size,
    "artifact_sha256": {
        path.name: sha256_file(path)
        for path in [
            history_file,
            patch_file,
            image_file,
            summary_file
        ]
    },
    "training_completed": True,
    "calibration_data_used": False,
    "test_data_used": False,
    "external_data_used": False
}

integrity_file = fold_dir / "checkpoint_integrity.json"

with open(integrity_file, "w", encoding="utf-8") as f:
    json.dump(integrity_record, f, indent=2)

print("\nCheckpoint SHA-256:")
print(checkpoint_hash)

# ------------------------------------------------------------
# 7. Ensure Fold 1 checkpoint was not modified
# ------------------------------------------------------------

fold1_checkpoint = Path(
    "/content/CIS631_model_checkpoints/"
    "efficientnet_b0/fold_1/"
    "efficientnet_b0_fold_1_best.pt"
)

expected_fold1_hash = (
    "387faf450467c89809df1c5f23d039d4"
    "ea5f0a77536f420d398c6013e035dd34"
)

if fold1_checkpoint.is_file():
    assert (
        sha256_file(fold1_checkpoint)
        == expected_fold1_hash
    )
    print("PASS: Fold 1 checkpoint unchanged.")
else:
    print(
        "NOTE: Fold 1 checkpoint not in this runtime; "
        "retain its previously downloaded backup."
    )

# ------------------------------------------------------------
# 8. Create ZIP backup
# ------------------------------------------------------------

backup_zip = Path(
    "/content/CIS631_EfficientNetB0_Fold2_Backup.zip"
)

with zipfile.ZipFile(
    backup_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for path in sorted(fold_dir.iterdir()):

        if path.is_file() and path.suffix in {
            ".csv", ".json"
        }:

            zf.write(
                path,
                arcname=str(path.relative_to(REPO_ROOT))
            )

    checkpoint_archive_name = (
        "checkpoints/efficientnet_b0/fold_2/"
        + checkpoint_file.name
    )

    zf.write(
        checkpoint_file,
        arcname=checkpoint_archive_name
    )

# ------------------------------------------------------------
# 9. Verify ZIP and checkpoint integrity
# ------------------------------------------------------------

with zipfile.ZipFile(backup_zip, "r") as zf:

    assert zf.testzip() is None

    assert checkpoint_archive_name in zf.namelist()

    archived_hash = hashlib.sha256()

    with zf.open(checkpoint_archive_name) as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            archived_hash.update(chunk)

    assert archived_hash.hexdigest() == checkpoint_hash

print("PASS: ZIP archive verified.")
print("PASS: Archived checkpoint SHA-256 matches.")

# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("EFFICIENTNET-B0 — FOLD 2 BACKUP")
print("=" * 65)

print("Selected epoch:", selected_epoch)
print("Validation AUROC:", round(metrics["auroc"], 6))

print(
    "Checkpoint size (MB):",
    round(checkpoint_file.stat().st_size / 1024**2, 2)
)

print(
    "Backup ZIP size (MB):",
    round(backup_zip.stat().st_size / 1024**2, 2)
)

print("\nBackup:", backup_zip)

print("\nPASS: Fold 2 metrics verified.")
print("PASS: Fold 2 checkpoint verified.")
print("PASS: Checkpoint integrity recorded.")
print("PASS: Backup ZIP created and verified.")
print("PASS: Fold 2 ready for archiving.")

# Download to local computer
files.download(str(backup_zip))


PASS: All Fold 2 artifacts located.
PASS: Training history and selection verified.
PASS: All six validation metrics reproduced.
Maximum aggregation difference: 1.1102230246251565e-16
PASS: Mean-logit aggregation verified.
PASS: Saved checkpoint loads correctly.

Checkpoint SHA-256:
53e7563ec89ff26a7e87c964c87d900cbd53e50c0c9584cc4128033e684bade9
PASS: Fold 1 checkpoint unchanged.
PASS: ZIP archive verified.
PASS: Archived checkpoint SHA-256 matches.

EFFICIENTNET-B0 — FOLD 2 BACKUP
Selected epoch: 5
Validation AUROC: 0.96942
Checkpoint size (MB): 15.57
Backup ZIP size (MB): 14.43

Backup: /content/CIS631_EfficientNetB0_Fold2_Backup.zip

PASS: Fold 2 metrics verified.
PASS: Fold 2 checkpoint verified.
PASS: Checkpoint integrity recorded.
PASS: Backup ZIP created and verified.
PASS: Fold 2 ready for archiving.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>